In [ ]:
# In[1]:

import pandas as pd
import pytz

tz = pytz.timezone('Asia/Shanghai')

# Load CSVs
metric_df = pd.read_csv('metric.csv')
trace_df = pd.read_csv('trace.csv')
log_df = pd.read_csv('log.csv')
error_df = pd.read_csv('error_logs.csv')

# Ensure timestamp and value types
for df in (metric_df, trace_df, log_df, error_df):
    if 'timestamp' in df.columns:
        df['timestamp'] = pd.to_numeric(df['timestamp'], errors='coerce').astype('Int64')

for df in (metric_df, trace_df, log_df):
    df['value'] = pd.to_numeric(df['value'], errors='coerce')

# Convert timestamps to timezone-aware datetime (UTC -> Asia/Shanghai)
for df in (metric_df, trace_df, log_df, error_df):
    df['ts_dt'] = pd.to_datetime(df['timestamp'].astype('int64'), unit='s', utc=True).dt.tz_convert(tz)

# 1) metric.csv aggregation
metric_grp = metric_df.groupby(['cmdb_id', 'kpi_name'], dropna=False)
metric_agg = metric_grp.agg(
    count=('value', 'count'),
    min_timestamp=('ts_dt', 'min'),
    max_timestamp=('ts_dt', 'max'),
)
# compute global P95 per group
metric_p95 = metric_grp['value'].quantile(0.95).rename('p95_value')
metric_agg = metric_agg.join(metric_p95).reset_index()
metric_agg = metric_agg.sort_values('count', ascending=False)
metric_agg_top = metric_agg.head(50)[['cmdb_id','kpi_name','count','min_timestamp','max_timestamp','p95_value']]

# 2) trace.csv aggregation
trace_grp = trace_df.groupby(['cmdb_id', 'trace_name'], dropna=False)
trace_agg = trace_grp.agg(
    count=('value', 'count'),
    min_timestamp=('ts_dt', 'min'),
    max_timestamp=('ts_dt', 'max'),
)
trace_p95 = trace_grp['value'].quantile(0.95).rename('p95_value')
trace_agg = trace_agg.join(trace_p95).reset_index()
trace_agg = trace_agg.sort_values('count', ascending=False)
trace_agg_top = trace_agg.head(50)[['cmdb_id','trace_name','count','min_timestamp','max_timestamp','p95_value']]

# 3) log.csv aggregation
log_grp = log_df.groupby(['cmdb_id', 'log_name'], dropna=False)
log_agg = log_grp.agg(
    count=('value', 'count'),
    min_timestamp=('ts_dt', 'min'),
    max_timestamp=('ts_dt', 'max'),
)
log_p95 = log_grp['value'].quantile(0.95).rename('p95_value')
log_agg = log_agg.join(log_p95).reset_index()
log_agg = log_agg.sort_values('count', ascending=False)
log_agg_top = log_agg.head(50)[['cmdb_id','log_name','count','min_timestamp','max_timestamp','p95_value']]

# 4) error_logs.csv aggregation: count and earliest 5 messages per cmdb_id
# Ensure message is string
error_df['message'] = error_df['message'].astype(str)

def sample_messages(subdf, n=5):
    s = subdf.sort_values('timestamp').head(n)['message'].tolist()
    return " || ".join(s)

error_group = error_df.groupby('cmdb_id', dropna=False)
error_agg = error_group.agg(
    count=('message', 'count'),
    earliest_timestamp=('ts_dt', 'min'),
).reset_index()

# build sample_messages column
sample_msgs = error_group.apply(lambda g: sample_messages(g, n=5)).rename('sample_messages').reset_index()
error_agg = error_agg.merge(sample_msgs, on='cmdb_id', how='left')
error_agg = error_agg.sort_values('count', ascending=False)
error_agg_top = error_agg.head(50)[['cmdb_id','count','earliest_timestamp','sample_messages']]

# Display the aggregated top tables (stored full tables in *_agg variables for later use)
metric_agg_top, trace_agg_top, log_agg_top, error_agg_top

```
Out[1]:
```
```python
# Summarize the previously computed aggregation tables in plain English.
# Uses in-memory variables: metric_agg_top, trace_agg_top, log_agg_top, error_agg_top

top_n = 3

# Prepare compact top entries for metric, trace, log
metric_rows = metric_agg_top.head(top_n)[['cmdb_id','kpi_name','count','p95_value']].to_dict('records')
trace_rows = trace_agg_top.head(top_n)[['cmdb_id','trace_name','count','p95_value']].to_dict('records')
log_rows = log_agg_top.head(top_n)[['cmdb_id','log_name','count','p95_value']].to_dict('records')

# Prepare top error log cmdbs (up to 5) with count and earliest timestamp
error_rows = error_agg_top.head(5)[['cmdb_id','count','earliest_timestamp','sample_messages']].to_dict('records')

# Build summary sentences
lines = []
lines.append("Summary of telemetry aggregates (compact):")

# Metric summary
lines.append(f"- metric.csv: showing top {top_n} (cmdb_id, kpi_name) by point count:")
for r in metric_rows:
    lines.append(f"  • {r['cmdb_id']} / {r['kpi_name']}: count={int(r['count'])}, global P95={r['p95_value']:.6g}")

# Trace summary
lines.append(f"- trace.csv: showing top {top_n} (cmdb_id, trace_name) by point count:")
for r in trace_rows:
    lines.append(f"  • {r['cmdb_id']} / {r['trace_name']}: count={int(r['count'])}, global P95={r['p95_value']:.6g}")

# Log summary
lines.append(f"- log.csv: showing top {top_n} (cmdb_id, log_name) by point count:")
for r in log_rows:
    lines.append(f"  • {r['cmdb_id']} / {r['log_name']}: count={int(r['count'])}, global P95={r['p95_value']:.6g}")

# Error logs summary
lines.append(f"- error_logs.csv: top {min(5, len(error_rows))} cmdb_id by error count with earliest timestamp and up to 5 sample messages:")
for r in error_rows:
    # truncate sample messages to keep summary compact
    sample = (r['sample_messages'] or "") 
    sample_snip = (sample[:200] + '...') if len(sample) > 200 else sample
    ts = r['earliest_timestamp']
    lines.append(f"  • {r['cmdb_id']}: count={int(r['count'])}, earliest={ts}, sample_messages=\"{sample_snip}\"")

lines.append("End of summary.")

summary = "\n".join(lines)
summary
```

The original code execution output of IPython Kernel is also provided below for reference:

(           cmdb_id                                           kpi_name  count             min_timestamp             max_timestamp     p95_value
1197      Tomcat02  Tomcat-Sessions_7441--log.Home_IS_UNDEFINED_SE...     31 2021-03-24 11:00:00+08:00 2021-03-24 11:30:00+08:00  0.000000e+00
3             IG01        JVM-Memory_7778_JVM_Memory_NoHeapMemoryUsed     31 2021-03-24 11:00:00+08:00 2021-03-24 11:30:00+08:00  5.850109e+07
6             IG01           JVM-Threads_7778_JVM_ThreadCount_Threads     31 2021-03-24 11:00:00+08:00 2021-03-24 11:30:00+08:00  7.600000e+01
1196      Tomcat02  Tomcat-Sessions_7441--log.Home_IS_UNDEFINED_SE...     31 2021-03-24 11:00:00+08:00 2021-03-24 11:30:00+08:00  0.000000e+00
1195      Tomcat02  Tomcat-Sessions_7441--log.Home_IS_UNDEFINED_SE...     31 2021-03-24 11:00:00+08:00 2021-03-24 11:30:00+08:00  0.000000e+00
...            ...                                                ...    ...                       ...                       ...           ...
1024  ServiceTest8                                             app.sr     31 2021-03-24 11:00:00+08:00 2021-03-24 11:30:00+08:00  1.000000e+02
1023  ServiceTest8                                             app.rr     31 2021-03-24 11:00:00+08:00 2021-03-24 11:30:00+08:00  1.000000e+02
1022  ServiceTest8                                            app.mrt     31 2021-03-24 11:00:00+08:00 2021-03-24 11:30:00+08:00  5.873040e+00
1261      Tomcat03         OSLinux-OSLinux_PROCESS_zabbix_PROCPPCount     31 2021-03-24 11:00:00+08:00 2021-03-24 11:30:00+08:00  3.000000e+00
1199      Tomcat02  Tomcat-Sessions_7441--logHome_IS_UNDEFINED_SES...     31 2021-03-24 11:00:00+08:00 2021-03-24 11:30:00+08:00  0.000000e+00

[50 rows x 6 columns],       cmdb_id                         trace_name  count             min_timestamp             max_timestamp   p95_value
257  dockerB2            trace.to_MG02.row_count     31 2021-03-24 11:00:00+08:00 2021-03-24 11:30:00+08:00  132.500000
0        IG01           trace.self.duration_mean     31 2021-03-24 11:00:00+08:00 2021-03-24 11:30:00+08:00    6.855119
218  dockerA2            trace.to_MG01.row_count     31 2021-03-24 11:00:00+08:00 2021-03-24 11:30:00+08:00  115.500000
222  dockerB1      trace.from_MG01.duration_mean     31 2021-03-24 11:00:00+08:00 2021-03-24 11:30:00+08:00    0.850616
223  dockerB1       trace.from_MG01.duration_p95     31 2021-03-24 11:00:00+08:00 2021-03-24 11:30:00+08:00    4.509300
..        ...                                ...    ...                       ...                       ...         ...
47       MG01      trace.from_dockerA2.row_count     31 2021-03-24 11:00:00+08:00 2021-03-24 11:30:00+08:00  115.500000
48       MG01  trace.from_dockerB1.duration_mean     31 2021-03-24 11:00:00+08:00 2021-03-24 11:30:00+08:00    0.162012
49       MG01   trace.from_dockerB1.duration_p95     31 2021-03-24 11:00:00+08:00 2021-03-24 11:30:00+08:00    1.022425
34       MG01   trace.from_Tomcat02.duration_p95     31 2021-03-24 11:00:00+08:00 2021-03-24 11:30:00+08:00    2.715450
35       MG01      trace.from_Tomcat02.row_count     31 2021-03-24 11:00:00+08:00 2021-03-24 11:30:00+08:00  299.500000

[50 rows x 6 columns],      cmdb_id          log_name  count             min_timestamp             max_timestamp  p95_value
4   Tomcat01     log.row_count     31 2021-03-24 11:00:00+08:00 2021-03-24 11:30:00+08:00   627.5000
2   Tomcat01   log.error_count     31 2021-03-24 11:00:00+08:00 2021-03-24 11:30:00+08:00     0.0000
16  apache02   log.error_count     31 2021-03-24 11:00:00+08:00 2021-03-24 11:30:00+08:00    35.5000
15  apache01     log.row_count     31 2021-03-24 11:00:00+08:00 2021-03-24 11:30:00+08:00  1181.0000
5   Tomcat02   log.error_count     31 2021-03-24 11:00:00+08:00 2021-03-24 11:30:00+08:00     0.0000
7   Tomcat02     log.row_count     31 2021-03-24 11:00:00+08:00 2021-03-24 11:30:00+08:00   613.5000
8   Tomcat03   log.error_count     31 2021-03-24 11:00:00+08:00 2021-03-24 11:30:00+08:00     0.0000
13  Tomcat04     log.row_count     31 2021-03-24 11:00:00+08:00 2021-03-24 11:30:00+08:00   581.0000
11  Tomcat04   log.error_count     31 2021-03-24 11:00:00+08:00 2021-03-24 11:30:00+08:00     0.0000
10  Tomcat03     log.row_count     31 2021-03-24 11:00:00+08:00 2021-03-24 11:30:00+08:00   575.0000
17  apache02     log.row_count     31 2021-03-24 11:00:00+08:00 2021-03-24 11:30:00+08:00  1194.5000
14  apache01   log.error_count     31 2021-03-24 11:00:00+08:00 2021-03-24 11:30:00+08:00    30.5000
1    Mysql01     log.row_count     27 2021-03-24 11:00:00+08:00 2021-03-24 11:29:00+08:00    76.4000
0    Mysql01   log.error_count     27 2021-03-24 11:00:00+08:00 2021-03-24 11:29:00+08:00     0.0000
12  Tomcat04  log.gc_pause_max     26 2021-03-24 11:00:00+08:00 2021-03-24 11:29:00+08:00     0.1100
3   Tomcat01  log.gc_pause_max     24 2021-03-24 11:00:00+08:00 2021-03-24 11:29:00+08:00     0.0900
6   Tomcat02  log.gc_pause_max     23 2021-03-24 11:01:00+08:00 2021-03-24 11:28:00+08:00     0.0900
9   Tomcat03  log.gc_pause_max     22 2021-03-24 11:00:00+08:00 2021-03-24 11:29:00+08:00     0.0895,     cmdb_id  count        earliest_timestamp                                    sample_messages
1  apache02    151 2021-03-24 11:08:00+08:00  IPAddress "POST /UOCP/person/ServiceTest9.json...
0  apache01    136 2021-03-24 11:08:04+08:00  IPAddress "POST /UOCP/person/ServiceTest9.json...)

**Note**: The printed pandas DataFrame is truncated due to its size. Only **10 rows** are displayed, which may introduce observation bias due to the incomplete table. If you want to comprehensively understand the details without bias, please ask Executor using `df.head(X)` to display more rows.```
```

In [ ]:
# In[3]:

import pandas as pd
import pytz
tz = pytz.timezone('Asia/Shanghai')

# Candidate components
candidates = ['apache01','apache02','Tomcat01','Tomcat02','Tomcat03','Tomcat04',
              'MG01','MG02','IG01','IG02','Mysql01','Mysql02','Redis01','Redis02']

# Ensure metric_df exists and types are correct (reusing existing variable)
metric_df['value'] = pd.to_numeric(metric_df['value'], errors='coerce')
metric_df['timestamp'] = pd.to_numeric(metric_df['timestamp'], errors='coerce').astype('Int64')
# Ensure ts_dt exists and is timezone-aware Asia/Shanghai
metric_df['ts_dt'] = pd.to_datetime(metric_df['timestamp'].astype('int64'), unit='s', utc=True).dt.tz_convert(tz)

# 2) Compute global P95 and P5 across the full series (do NOT filter by time)
global_p95 = metric_df.groupby(['cmdb_id','kpi_name'], dropna=False)['value'].quantile(0.95).rename('global_P95').reset_index()
global_p5  = metric_df.groupby(['cmdb_id','kpi_name'], dropna=False)['value'].quantile(0.05).rename('global_P5').reset_index()
global_thresh = pd.merge(global_p95, global_p5, on=['cmdb_id','kpi_name'], how='outer')

# Filter thresholds to candidate components only (we only analyze these)
global_thresh = global_thresh[global_thresh['cmdb_id'].isin(candidates)].reset_index(drop=True)

# 3) Filter metric points to the failure window (inclusive)
start = pd.Timestamp('2021-03-24 11:00:00', tz=tz)
end   = pd.Timestamp('2021-03-24 11:30:00', tz=tz)
window_df = metric_df[
    metric_df['cmdb_id'].isin(candidates) &
    (metric_df['ts_dt'] >= start) &
    (metric_df['ts_dt'] <= end)
].copy()

# Merge global thresholds into window rows
window_with_thresh = pd.merge(window_df, global_thresh, on=['cmdb_id','kpi_name'], how='left')

# 3) For each (cmdb_id, kpi_name) in window compute required stats
rows = []
grouped = window_with_thresh.groupby(['cmdb_id','kpi_name'], sort=False)
for (cmdb_id, kpi_name), g in grouped:
    gp95 = g['global_P95'].iloc[0]
    gp5  = g['global_P5'].iloc[0]
    vals = g['value']
    # count high/low
    count_high = int((vals >= gp95).sum()) if pd.notna(gp95) else 0
    count_low  = int((vals <= gp5).sum()) if pd.notna(gp5) else 0
    earliest_high_ts = g.loc[vals >= gp95, 'ts_dt'].min() if count_high>0 else pd.NaT
    earliest_low_ts  = g.loc[vals <= gp5, 'ts_dt'].min() if count_low>0 else pd.NaT
    max_value_in_window = vals.max()
    min_value_in_window = vals.min()
    # severity computations, follow null rules
    if pd.notna(gp95) and gp95 != 0:
        severity_high = max_value_in_window / gp95
    else:
        severity_high = pd.NA
    if pd.notna(min_value_in_window) and min_value_in_window != 0 and pd.notna(gp5):
        severity_low = gp5 / min_value_in_window
    else:
        severity_low = pd.NA

    rows.append({
        'cmdb_id': cmdb_id,
        'kpi_name': kpi_name,
        'global_P95': gp95,
        'global_P5': gp5,
        'count_high': count_high,
        'earliest_high_timestamp': earliest_high_ts,
        'max_value_in_window': max_value_in_window,
        'severity_high': severity_high,
        'count_low': count_low,
        'earliest_low_timestamp': earliest_low_ts,
        'min_value_in_window': min_value_in_window,
        'severity_low': severity_low
    })

metric_anom_full = pd.DataFrame(rows)

# 4) Keep only rows where count_high>0 or count_low>0
if not metric_anom_full.empty:
    metric_anom_full = metric_anom_full[(metric_anom_full['count_high']>0) | (metric_anom_full['count_low']>0)].reset_index(drop=True)

# 5) Sort by descending max(severity_high, severity_low) and limit top 20
# For sorting, treat pd.NA as 0
def safe_val(x):
    return 0 if (pd.isna(x) or x is pd.NA) else x

if not metric_anom_full.empty:
    metric_anom_full['severity_high_num'] = metric_anom_full['severity_high'].apply(safe_val).astype(float)
    metric_anom_full['severity_low_num']  = metric_anom_full['severity_low'].apply(safe_val).astype(float)
    metric_anom_full['severity_max'] = metric_anom_full[['severity_high_num','severity_low_num']].max(axis=1)
    metric_anom_full = metric_anom_full.sort_values('severity_max', ascending=False).reset_index(drop=True)
else:
    # ensure consistent columns if empty
    cols = ['cmdb_id','kpi_name','global_P95','global_P5','count_high','earliest_high_timestamp',
            'max_value_in_window','severity_high','count_low','earliest_low_timestamp','min_value_in_window','severity_low']
    metric_anom_full = pd.DataFrame(columns=cols)

# Prepare top 20 display table with requested columns
display_cols = ['cmdb_id','kpi_name','global_P95','global_P5',
                'count_high','earliest_high_timestamp','max_value_in_window','severity_high',
                'count_low','earliest_low_timestamp','min_value_in_window','severity_low']
metric_anom_top = metric_anom_full[display_cols].head(20)

# Display only the limited top 20 rows as requested
metric_anom_top

```
Out[3]:
```
```python
# Summarize metric-based anomaly detection results in plain English.
# Uses in-memory variable: metric_anom_full (full anomaly table) and metric_anom_top (displayed top table)

top_n = 5

total_anomalies = int(metric_anom_full.shape[0]) if 'metric_anom_full' in globals() else 0

summary_lines = []
summary_lines.append(f"Metric-based anomaly detection summary (UTC+8 failure window 2021-03-24 11:00–11:30):")
summary_lines.append(f"- Total series with anomalies (count_high>0 or count_low>0): {total_anomalies}")
if total_anomalies == 0:
    summary_lines.append("- No anomalies detected for the candidate components in the window.")
else:
    summary_lines.append(f"- Top {min(top_n, total_anomalies)} anomalous series by severity:")
    top_rows = metric_anom_full.head(top_n)[['cmdb_id','kpi_name','severity_max','max_value_in_window','global_P95','earliest_high_timestamp','count_high','count_low']]
    for i, r in top_rows.iterrows():
        sev = float(r['severity_max']) if pd.notna(r['severity_max']) else None
        sev_str = f"{sev:.3f}" if sev is not None else "NA"
        summary_lines.append(
            f"  {i+1}. {r['cmdb_id']} / {r['kpi_name']} — severity={sev_str}; "
            f"max_in_window={r['max_value_in_window']}; global_P95={r['global_P95']}; "
            f"earliest_high={r['earliest_high_timestamp']}; count_high={int(r['count_high'])}; count_low={int(r['count_low'])}"
        )

    summary_lines.append("- Notes: severity = max(value)/global_P95 for high-side anomalies or global_P5/min(value) for low-side;")
    summary_lines.append("  only series with at least one high or low event in the window are included; full anomaly table is stored in variable metric_anom_full.")

summary = "\n".join(summary_lines)
summary
```

The original code execution output of IPython Kernel is also provided below for reference:

cmdb_id                                          kpi_name  global_P95  global_P5  count_high   earliest_high_timestamp  max_value_in_window severity_high  count_low    earliest_low_timestamp  min_value_in_window severity_low
0   Tomcat03                   OSLinux-CPU_CPU-2_SingleCpuidle   98.384432  20.167636           1 2021-03-24 11:11:00+08:00            98.752287      1.003739          1 2021-03-24 11:01:00+08:00             3.339425     6.039254
1   Tomcat01    OSLinux-OSLinux_LOCALDISK_LOCALDISK-sda_DSKBps    0.300000   0.000000           1 2021-03-24 11:07:00+08:00             1.000000      3.333333         14 2021-03-24 11:00:00+08:00             0.000000         <NA>
2       MG02                   OSLinux-CPU_CPU-2_SingleCpuidle    4.743100   0.000000           1 2021-03-24 11:07:00+08:00            13.551713      2.857143         13 2021-03-24 11:01:00+08:00             0.000000         <NA>
3   Tomcat03   OSLinux-OSLinux_LOCALDISK_LOCALDISK-sdb_DSKRead    0.046667   0.000000           1 2021-03-24 11:21:00+08:00             0.133333      2.857143         13 2021-03-24 11:01:00+08:00             0.000000         <NA>
4   Tomcat03   OSLinux-OSLinux_LOCALDISK_LOCALDISK-sdb_DSKRTps    0.005833   0.000000           1 2021-03-24 11:21:00+08:00             0.016667      2.857143         13 2021-03-24 11:01:00+08:00             0.000000         <NA>
5   Tomcat02    OSLinux-OSLinux_LOCALDISK_LOCALDISK-sda_DSKBps    1.050000   0.000000           1 2021-03-24 11:17:00+08:00             3.000000      2.857143         13 2021-03-24 11:01:00+08:00             0.000000         <NA>
6   Tomcat03   OSLinux-OSLinux_LOCALDISK_LOCALDISK-sda_DSKRead  266.053333   0.000000           1 2021-03-24 11:15:00+08:00           748.266667      2.812469          7 2021-03-24 11:07:00+08:00             0.000000         <NA>
7   apache02                            OSLinux-CPU_CPU_CPUWio    1.097300   0.044650           1 2021-03-24 11:12:00+08:00             2.407100      2.193657          1 2021-03-24 11:20:00+08:00             0.016600     2.689759
8   Tomcat03   OSLinux-OSLinux_LOCALDISK_LOCALDISK-sda_DSKRTps    1.688333   0.000000           1 2021-03-24 11:15:00+08:00             4.483333      2.655479          7 2021-03-24 11:07:00+08:00             0.000000         <NA>
9   Tomcat01                            OSLinux-CPU_CPU_CPUWio    1.362466   0.024959           2 2021-03-24 11:13:00+08:00             3.432494      2.519324          2 2021-03-24 11:09:00+08:00             0.020799     1.200024
10  Tomcat03                            OSLinux-CPU_CPU_CPUWio    2.597648   0.056163           2 2021-03-24 11:13:00+08:00             5.775152      2.223223          2 2021-03-24 11:09:00+08:00             0.045776     1.226898
11   Redis02    OSLinux-OSLinux_LOCALDISK_LOCALDISK-sda_DSKBps    1.900000   0.000000           1 2021-03-24 11:27:00+08:00             4.000000      2.105263         13 2021-03-24 11:00:00+08:00             0.000000         <NA>
12  Tomcat03    OSLinux-OSLinux_LOCALDISK_LOCALDISK-sda_DSKBps    2.400000   0.000000           1 2021-03-24 11:07:00+08:00             5.000000      2.083333         11 2021-03-24 11:03:00+08:00             0.000000         <NA>
13      MG02    OSLinux-OSLinux_LOCALDISK_LOCALDISK-sda_DSKBps    2.400000   0.000000           1 2021-03-24 11:07:00+08:00             5.000000      2.083333         12 2021-03-24 11:01:00+08:00             0.000000         <NA>
14      MG02   OSLinux-OSLinux_LOCALDISK_LOCALDISK-sda_DSKRead    3.106666   0.000000           1 2021-03-24 11:21:00+08:00             6.400000      2.060086         10 2021-03-24 11:05:00+08:00             0.000000         <NA>
15  Tomcat03  OSLinux-OSLinux_LOCALDISK_LOCALDISK-sda_DSKWrite   32.326667   7.553333           1 2021-03-24 11:15:00+08:00            65.866667      2.037534          1 2021-03-24 11:05:00+08:00             7.466667     1.011607
16  Tomcat02   OSLinux-OSLinux_LOCALDISK_LOCALDISK-sda_DSKRTps    0.188333   0.000000           1 2021-03-24 11:17:00+08:00             0.383333        2.0354         11 2021-03-24 11:03:00+08:00             0.000000         <NA>
17  Tomcat02   OSLinux-OSLinux_LOCALDISK_LOCALDISK-sda_DSKRead    8.926667   0.000000           1 2021-03-24 11:05:00+08:00            17.333333      1.941748         11 2021-03-24 11:03:00+08:00             0.000000         <NA>
18  apache01                   OSLinux-CPU_CPU-2_SingleCpuidle   99.561030  46.639970           1 2021-03-24 11:11:00+08:00            99.600900        1.0004          1 2021-03-24 11:13:00+08:00            24.070400     1.937648
19   Redis02                            OSLinux-CPU_CPU_CPUWio    0.549616   0.004200           2 2021-03-24 11:14:00+08:00             1.062177      1.932581          5 2021-03-24 11:07:00+08:00             0.004192     1.001908```
```

In [ ]:
# In[4]:

import pandas as pd
import pytz

# reuse timezone and dataframes from kernel if available
tz = pytz.timezone('Asia/Shanghai')

# Candidate components
candidates = ['Tomcat03','Tomcat02','Tomcat01','apache01','apache02','MG02','Redis02','IG01']

# Ensure trace_df exists and types are correct (reuse variable)
trace_df['value'] = pd.to_numeric(trace_df['value'], errors='coerce')
trace_df['timestamp'] = pd.to_numeric(trace_df['timestamp'], errors='coerce').astype('Int64')
trace_df['ts_dt'] = pd.to_datetime(trace_df['timestamp'].astype('int64'), unit='s', utc=True).dt.tz_convert(tz)

# 1) Compute global P95 and P5 across the full trace series for candidate components
trace_thresh_p95 = trace_df.groupby(['cmdb_id','trace_name'], dropna=False)['value'].quantile(0.95).rename('global_P95').reset_index()
trace_thresh_p5  = trace_df.groupby(['cmdb_id','trace_name'], dropna=False)['value'].quantile(0.05).rename('global_P5').reset_index()
trace_thresh = pd.merge(trace_thresh_p95, trace_thresh_p5, on=['cmdb_id','trace_name'], how='outer')

# Restrict thresholds to candidate components
trace_thresh = trace_thresh[trace_thresh['cmdb_id'].isin(candidates)].reset_index(drop=True)

# 2) Filter trace points to the failure window (inclusive)
start = pd.Timestamp('2021-03-24 11:00:00', tz=tz)
end   = pd.Timestamp('2021-03-24 11:30:00', tz=tz)
trace_window = trace_df[
    trace_df['cmdb_id'].isin(candidates) &
    (trace_df['ts_dt'] >= start) &
    (trace_df['ts_dt'] <= end)
].copy()

# Merge global thresholds into window rows
trace_window = pd.merge(trace_window, trace_thresh, on=['cmdb_id','trace_name'], how='left')

# 3) For each (cmdb_id, trace_name) in window compute required stats
rows = []
grouped = trace_window.groupby(['cmdb_id','trace_name'], sort=False)
for (cmdb_id, trace_name), g in grouped:
    gp95 = g['global_P95'].iloc[0]
    gp5  = g['global_P5'].iloc[0]
    vals = g['value']
    # counts
    count_high = int((vals >= gp95).sum()) if pd.notna(gp95) else 0
    count_low  = int((vals <= gp5).sum()) if pd.notna(gp5) else 0
    earliest_high_ts = g.loc[vals >= gp95, 'ts_dt'].min() if count_high>0 else pd.NaT
    earliest_low_ts  = g.loc[vals <= gp5, 'ts_dt'].min() if count_low>0 else pd.NaT
    max_value_in_window = vals.max()
    min_value_in_window = vals.min()
    # severity calculations with null rules
    if pd.notna(gp95) and gp95 != 0:
        severity_high = max_value_in_window / gp95
    else:
        severity_high = pd.NA
    if pd.notna(min_value_in_window) and min_value_in_window != 0 and pd.notna(gp5):
        severity_low = gp5 / min_value_in_window
    else:
        severity_low = pd.NA

    rows.append({
        'cmdb_id': cmdb_id,
        'trace_name': trace_name,
        'global_P95': gp95,
        'global_P5': gp5,
        'count_high': count_high,
        'earliest_high_timestamp': earliest_high_ts,
        'max_value_in_window': max_value_in_window,
        'severity_high': severity_high,
        'count_low': count_low,
        'earliest_low_timestamp': earliest_low_ts,
        'min_value_in_window': min_value_in_window,
        'severity_low': severity_low
    })

trace_anom_full = pd.DataFrame(rows)

# 4) Keep only rows where count_high>0 or count_low>0
if not trace_anom_full.empty:
    trace_anom_full = trace_anom_full[(trace_anom_full['count_high']>0) | (trace_anom_full['count_low']>0)].reset_index(drop=True)

# 5) Sort by descending max(severity_high, severity_low) and limit output to top 20
def safe_num(x):
    return 0.0 if (pd.isna(x) or x is pd.NA) else float(x)

if not trace_anom_full.empty:
    trace_anom_full['severity_high_num'] = trace_anom_full['severity_high'].apply(safe_num)
    trace_anom_full['severity_low_num']  = trace_anom_full['severity_low'].apply(safe_num)
    trace_anom_full['severity_max'] = trace_anom_full[['severity_high_num','severity_low_num']].max(axis=1)
    trace_anom_full = trace_anom_full.sort_values('severity_max', ascending=False).reset_index(drop=True)
else:
    # ensure consistent columns if empty
    cols = ['cmdb_id','trace_name','global_P95','global_P5','count_high','earliest_high_timestamp',
            'max_value_in_window','severity_high','count_low','earliest_low_timestamp','min_value_in_window','severity_low']
    trace_anom_full = pd.DataFrame(columns=cols)

display_cols = ['cmdb_id','trace_name','global_P95','global_P5',
                'count_high','earliest_high_timestamp','max_value_in_window','severity_high',
                'count_low','earliest_low_timestamp','min_value_in_window','severity_low']

trace_anom_top = trace_anom_full[display_cols].head(20)

# Display only the limited top 20 rows as requested (trace_anom_full stored for later use)
trace_anom_top

```
Out[4]:
```
```python
# Summarize the trace-based anomaly detection results in plain English.
# Uses in-memory variable: trace_anom_full (full anomaly table) and trace_anom_top (displayed top table)

import pandas as pd

if 'trace_anom_full' not in globals() or trace_anom_full.empty:
    summary = "No trace anomalies detected for the selected components in the given failure window."
else:
    total = int(trace_anom_full.shape[0])
    top_n = min(5, total)
    # Ensure severity_max exists; it was computed earlier
    if 'severity_max' not in trace_anom_full.columns:
        trace_anom_full['severity_high_num'] = trace_anom_full['severity_high'].apply(lambda x: 0.0 if (pd.isna(x) or x is pd.NA) else float(x))
        trace_anom_full['severity_low_num']  = trace_anom_full['severity_low'].apply(lambda x: 0.0 if (pd.isna(x) or x is pd.NA) else float(x))
        trace_anom_full['severity_max'] = trace_anom_full[['severity_high_num','severity_low_num']].max(axis=1)
        trace_anom_full = trace_anom_full.sort_values('severity_max', ascending=False).reset_index(drop=True)
    top_rows = trace_anom_full.head(top_n)

    lines = []
    lines.append(f"Trace-based anomaly detection summary for window 2021-03-24 11:00–11:30 (UTC+8):")
    lines.append(f"- Total trace series with anomalies (count_high>0 or count_low>0): {total}")
    lines.append(f"- Top {top_n} anomalous trace series by severity:")

    for i, r in top_rows.iterrows():
        sev = float(r['severity_max']) if pd.notna(r['severity_max']) else None
        sev_str = f"{sev:.3f}" if sev is not None else "NA"
        eh = r['earliest_high_timestamp']
        el = r['earliest_low_timestamp']
        # Format timestamps concisely
        eh_s = eh.isoformat() if (not pd.isna(eh) and hasattr(eh, 'isoformat')) else "None"
        el_s = el.isoformat() if (not pd.isna(el) and hasattr(el, 'isoformat')) else "None"
        lines.append(
            f"  {i+1}. {r['cmdb_id']} / {r['trace_name']} — severity_max={sev_str}; "
            f"max_in_window={r['max_value_in_window']}; global_P95={r['global_P95']}; "
            f"earliest_high={eh_s}; count_high={int(r['count_high'])}; count_low={int(r['count_low'])}; earliest_low={el_s}"
        )

    lines.append("- Observations: many top anomalies are high-side spikes in row_count metrics between services (e.g., trace.from_*/trace.to_* and trace.self.row_count), with severity_max values modestly above 1.0 (≈1.03–1.13).")
    lines.append("- Some traces also show low-side events (count_low>0) where values dropped to very small minima, producing large severity_low when global_P5 is nonzero.")
    lines.append("- Full anomaly table is stored in variable 'trace_anom_full' for further inspection.")

    summary = "\n".join(lines)

summary
```

The original code execution output of IPython Kernel is also provided below for reference:

cmdb_id                      trace_name   global_P95    global_P5  count_high   earliest_high_timestamp  max_value_in_window  severity_high  count_low    earliest_low_timestamp  min_value_in_window  severity_low
0   Tomcat02       trace.from_IG02.row_count   302.000000   180.500000           2 2021-03-24 11:06:00+08:00             332.0000       1.099338          2 2021-03-24 11:09:00+08:00              1.00000    180.500000
1   Tomcat02         trace.to_MG01.row_count   299.500000   169.500000           2 2021-03-24 11:05:00+08:00             326.0000       1.088481          2 2021-03-24 11:09:00+08:00              1.00000    169.500000
2   Tomcat01       trace.from_IG02.row_count   307.500000   187.000000           2 2021-03-24 11:00:00+08:00             323.0000       1.050407          3 2021-03-24 11:09:00+08:00              2.00000     93.500000
3       MG02   trace.from_Tomcat04.row_count   305.000000   184.500000           2 2021-03-24 11:02:00+08:00             314.0000       1.029508          2 2021-03-24 11:10:00+08:00              2.00000     92.250000
4   Tomcat03         trace.to_MG01.row_count   284.500000    91.500000           2 2021-03-24 11:05:00+08:00             324.0000       1.138840          2 2021-03-24 11:10:00+08:00              1.00000     91.500000
5       IG01     trace.to_Tomcat02.row_count   316.500000   177.000000           2 2021-03-24 11:05:00+08:00             327.0000       1.033175          2 2021-03-24 11:10:00+08:00              2.00000     88.500000
6   Tomcat02       trace.from_IG01.row_count   316.500000   177.000000           2 2021-03-24 11:05:00+08:00             327.0000       1.033175          2 2021-03-24 11:10:00+08:00              2.00000     88.500000
7   Tomcat01         trace.to_MG01.row_count   325.000000   174.500000           2 2021-03-24 11:00:00+08:00             339.0000       1.043077          2 2021-03-24 11:09:00+08:00              2.00000     87.250000
8   Tomcat02            trace.self.row_count  4181.500000  2376.500000           2 2021-03-24 11:05:00+08:00            4315.0000       1.031926          2 2021-03-24 11:10:00+08:00             30.00000     79.216667
9   Tomcat01            trace.self.row_count  4233.500000  2430.000000           2 2021-03-24 11:00:00+08:00            4284.0000       1.011929          2 2021-03-24 11:10:00+08:00             36.00000     67.500000
10      MG02   trace.from_Tomcat02.row_count   318.000000   189.000000           3 2021-03-24 11:01:00+08:00             357.0000       1.122642          2 2021-03-24 11:10:00+08:00              3.00000     63.000000
11  Tomcat02         trace.to_MG02.row_count   318.000000   189.000000           3 2021-03-24 11:01:00+08:00             357.0000       1.122642          2 2021-03-24 11:10:00+08:00              3.00000     63.000000
12      IG01     trace.to_Tomcat01.row_count   319.000000   175.000000           2 2021-03-24 11:00:00+08:00             337.0000       1.056426          2 2021-03-24 11:09:00+08:00              3.00000     58.333333
13  Tomcat01       trace.from_IG01.row_count   319.000000   175.000000           2 2021-03-24 11:00:00+08:00             337.0000       1.056426          2 2021-03-24 11:09:00+08:00              3.00000     58.333333
14      IG01  trace.to_Tomcat02.duration_p95     4.028525     1.575925           2 2021-03-24 11:12:00+08:00               4.3396       1.077218          2 2021-03-24 11:26:00+08:00              0.02715     58.045120
15  Tomcat02    trace.from_IG01.duration_p95     4.028525     1.575925           2 2021-03-24 11:12:00+08:00               4.3396       1.077218          2 2021-03-24 11:26:00+08:00              0.02715     58.045120
16  Tomcat01    trace.from_IG01.duration_p95     4.154100     1.417575           2 2021-03-24 11:13:00+08:00               5.0960       1.226740          2 2021-03-24 11:10:00+08:00              0.02470     57.391700
17      IG01  trace.to_Tomcat01.duration_p95     4.154100     1.417575           2 2021-03-24 11:13:00+08:00               5.0960       1.226740          2 2021-03-24 11:10:00+08:00              0.02470     57.391700
18      MG02     trace.to_dockerA1.row_count   122.000000    50.000000           2 2021-03-24 11:00:00+08:00             135.0000       1.106557          2 2021-03-24 11:10:00+08:00              1.00000     50.000000
19      IG01            trace.self.row_count  2422.000000  1415.000000           2 2021-03-24 11:05:00+08:00            2640.0000       1.090008          2 2021-03-24 11:10:00+08:00             30.00000     47.166667```
```

In [ ]:
# In[5]:

import pandas as pd
import pytz

tz = pytz.timezone('Asia/Shanghai')

# Reuse existing dataframes if available; otherwise load
try:
    log_df
except NameError:
    log_df = pd.read_csv('log.csv')
try:
    error_df
except NameError:
    error_df = pd.read_csv('error_logs.csv')

# Ensure types and timezone-aware datetime column (UTC -> Asia/Shanghai)
for df in (log_df, error_df):
    if 'timestamp' in df.columns:
        df['timestamp'] = pd.to_numeric(df['timestamp'], errors='coerce').astype('Int64')
        df['ts_dt'] = pd.to_datetime(df['timestamp'].astype('int64'), unit='s', utc=True).dt.tz_convert(tz)
    else:
        df['ts_dt'] = pd.NaT

# Failure window (inclusive) in UTC+8
start = pd.Timestamp('2021-03-24 11:00:00', tz=tz)
end   = pd.Timestamp('2021-03-24 11:30:00', tz=tz)

# 1) log.csv rows for specified components within the window
log_components = ['Tomcat01','Tomcat02','Tomcat03','Tomcat04','apache01','apache02','IG01','MG02']
log_window_full = log_df[
    log_df['cmdb_id'].isin(log_components) &
    (log_df['ts_dt'] >= start) & (log_df['ts_dt'] <= end)
].copy()

# Prepare output columns: timestamp (human-readable UTC+8), cmdb_id, log_name, value
log_window = log_window_full.assign(
    timestamp=log_window_full['ts_dt']  # use tz-aware datetime as "timestamp"
)[['timestamp','cmdb_id','log_name','value']].sort_values('timestamp', ascending=True).reset_index(drop=True)

# 2) error_logs.csv rows for specified components within the window
error_components = ['apache01','apache02','Tomcat01','Tomcat02','Tomcat03','IG01']
error_logs_full = error_df[
    error_df['cmdb_id'].isin(error_components) &
    (error_df['ts_dt'] >= start) & (error_df['ts_dt'] <= end)
].copy()

error_logs_window = error_logs_full.assign(
    timestamp=error_logs_full['ts_dt']
)[['timestamp','cmdb_id','message']].sort_values('timestamp', ascending=True).reset_index(drop=True)

# Store full returned tables in kernel variables (as requested)
# log_window and error_logs_window already assigned

# Display only the limited top 50 rows each
log_window.head(50), error_logs_window.head(50)

```
Out[5]:
```
```python
# Summarize log and error log window extracts in plain English.
# Uses in-memory variables: log_window, error_logs_window

import pandas as pd

# Safety checks
if 'log_window' not in globals() or 'error_logs_window' not in globals():
    summary = "Required window tables (log_window, error_logs_window) are not available in the kernel."
else:
    lw = log_window
    ew = error_logs_window

    # Basic counts and time ranges
    n_logs = int(lw.shape[0])
    n_errors = int(ew.shape[0])
    lw_earliest = lw['timestamp'].min() if n_logs>0 else None
    lw_latest   = lw['timestamp'].max() if n_logs>0 else None
    ew_earliest = ew['timestamp'].min() if n_errors>0 else None
    ew_latest   = ew['timestamp'].max() if n_errors>0 else None

    # Top components by row count in log_window (up to 5)
    top_log_comps = lw['cmdb_id'].value_counts().head(5)
    top_log_list = [f"{c}={int(v)}" for c,v in top_log_comps.items()]

    # Top components in error logs (up to 5)
    top_err_comps = ew['cmdb_id'].value_counts().head(5)
    top_err_list = [f"{c}={int(v)}" for c,v in top_err_comps.items()]

    # Detect dominant error time cluster (most frequent minute)
    if n_errors>0:
        # extract minute bucket string
        minute_counts = ew['timestamp'].dt.strftime('%Y-%m-%d %H:%M').value_counts()
        top_minute = minute_counts.idxmax()
        top_minute_count = int(minute_counts.max())
        # most frequent error-producing cmdb in that minute (if any)
        minute_mask = ew['timestamp'].dt.strftime('%Y-%m-%d %H:%M') == top_minute
        top_cmdb_in_minute = ew[minute_mask]['cmdb_id'].value_counts().idxmax()
    else:
        top_minute = None
        top_minute_count = 0
        top_cmdb_in_minute = None

    lines = [
        "Log-level correlation extract (window 2021-03-24 11:00–11:30, UTC+8):",
        f"- log.csv rows returned: {n_logs}. Time range: {lw_earliest} → {lw_latest}.",
        f"  Top components in these log rows (top 5): {', '.join(top_log_list) if top_log_list else 'none'}.",
        f"- error_logs.csv rows returned: {n_errors}. Time range: {ew_earliest} → {ew_latest}.",
        f"  Top components in error logs (top 5): {', '.join(top_err_list) if top_err_list else 'none'}.",
    ]

    if top_minute:
        lines.append(f"- Error messages are concentrated around minute {top_minute}: {top_minute_count} messages; most from {top_cmdb_in_minute}.")
    else:
        lines.append("- No error messages found in the window.")

    lines.append("- Notes: apache01/apache02 show high row_count values in the window; apache02 produces many error log messages around 11:08 (see error samples). Tomcat instances have log.error_count mostly zero in the displayed rows.")
    lines.append("- Full tables are stored in variables 'log_window' and 'error_logs_window' for further inspection.")

    summary = "\n".join(lines)

summary
```

The original code execution output of IPython Kernel is also provided below for reference:

(                   timestamp   cmdb_id         log_name   value
0  2021-03-24 11:00:00+08:00  Tomcat01  log.error_count     0.0
1  2021-03-24 11:00:00+08:00  apache02    log.row_count  1198.0
2  2021-03-24 11:00:00+08:00  apache02  log.error_count     0.0
3  2021-03-24 11:00:00+08:00  apache01    log.row_count  1166.0
4  2021-03-24 11:00:00+08:00  apache01  log.error_count     0.0
..                       ...       ...              ...     ...
45 2021-03-24 11:02:00+08:00  Tomcat03    log.row_count   498.0
46 2021-03-24 11:03:00+08:00  Tomcat04  log.error_count     0.0
47 2021-03-24 11:03:00+08:00  apache02    log.row_count  1015.0
48 2021-03-24 11:03:00+08:00  apache02  log.error_count     0.0
49 2021-03-24 11:03:00+08:00  apache01    log.row_count  1018.0

[50 rows x 4 columns],                    timestamp   cmdb_id                                            message
0  2021-03-24 11:08:00+08:00  apache02  IPAddress "POST /UOCP/person/ServiceTest9.json...
1  2021-03-24 11:08:01+08:00  apache02  IPAddress "POST /UOCP/base/ServiceTest6.json H...
2  2021-03-24 11:08:04+08:00  apache01  IPAddress "POST /UOCP/person/ServiceTest9.json...
3  2021-03-24 11:08:05+08:00  apache02  IPAddress "POST /UOCP/base/ServiceTest7.json H...
4  2021-03-24 11:08:05+08:00  apache02  IPAddress "POST /UOCP/person/ServiceTest3.json...
..                       ...       ...                                                ...
45 2021-03-24 11:08:38+08:00  apache02  IPAddress "POST /UOCP/person/ServiceTest5.json...
46 2021-03-24 11:08:38+08:00  apache02  IPAddress "POST /UOCP/person/ServiceTest10.jso...
47 2021-03-24 11:08:38+08:00  apache02  IPAddress "POST /UOCP/person/ServiceTest5.json...
48 2021-03-24 11:08:38+08:00  apache02  IPAddress "POST /UOCP/base/ServiceTest6.json H...
49 2021-03-24 11:08:40+08:00  apache02  IPAddress "POST /UOCP/person/ServiceTest3.json...

[50 rows x 3 columns])

**Note**: The printed pandas DataFrame is truncated due to its size. Only **10 rows** are displayed, which may introduce observation bias due to the incomplete table. If you want to comprehensively understand the details without bias, please ask Executor using `df.head(X)` to display more rows.```
```

In [ ]:
# In[6]:

import pandas as pd
import pytz

tz = pytz.timezone('Asia/Shanghai')

# Reuse existing metric_df from kernel; if not present, load
try:
    metric_df
except NameError:
    metric_df = pd.read_csv('metric.csv')

# Ensure types and timezone-aware datetime (UTC -> Asia/Shanghai)
metric_df['value'] = pd.to_numeric(metric_df['value'], errors='coerce')
metric_df['timestamp'] = pd.to_numeric(metric_df['timestamp'], errors='coerce').astype('Int64')
metric_df['ts_dt'] = pd.to_datetime(metric_df['timestamp'].astype('int64'), unit='s', utc=True).dt.tz_convert(tz)

# Target components
targets = ['Tomcat01', 'Tomcat03']

# 1) Discover KPI names for the two components containing 'LOCALDISK' or 'DSK' (case-sensitive substring match as requested)
mask_comp = metric_df['cmdb_id'].isin(targets)
mask_name = metric_df['kpi_name'].str.contains('LOCALDISK|DSK', na=False)
kpis_found = metric_df[mask_comp & mask_name][['cmdb_id','kpi_name']].drop_duplicates().reset_index(drop=True)

# 2) Compute global P95 per (cmdb_id, kpi_name) using the full series (do NOT filter by time)
# We must compute thresholds using the full metric_df but restricted to the discovered series.
# Build a DataFrame of unique (cmdb_id,kpi_name) pairs to compute thresholds only for them.
pairs = kpis_found.copy()
if not pairs.empty:
    # Merge to get all rows for these pairs from full metric_df
    merged = metric_df.merge(pairs, on=['cmdb_id','kpi_name'], how='inner')
    global_p95 = merged.groupby(['cmdb_id','kpi_name'], sort=False)['value'].quantile(0.95).rename('global_P95').reset_index()
else:
    global_p95 = pd.DataFrame(columns=['cmdb_id','kpi_name','global_P95'])

# 3) Filter metric points for these series to the failure window inclusive
start = pd.Timestamp('2021-03-24 11:00:00', tz=tz)
end   = pd.Timestamp('2021-03-24 11:30:00', tz=tz)

window_mask = (metric_df['ts_dt'] >= start) & (metric_df['ts_dt'] <= end)
window_points = metric_df[window_mask].merge(pairs, on=['cmdb_id','kpi_name'], how='inner').copy()

# Merge global_P95 into window_points
window_points = window_points.merge(global_p95, on=['cmdb_id','kpi_name'], how='left')

# For each series compute earliest_high_timestamp (first ts where value >= global_P95), max_value_in_window, count_high
rows = []
grouped = window_points.groupby(['cmdb_id','kpi_name'], sort=False)
for (cmdb_id, kpi_name), g in grouped:
    gp95 = g['global_P95'].iloc[0]
    vals = g['value']
    # count_high and earliest timestamp where value >= gp95 (respect NaN gp95)
    if pd.notna(gp95):
        mask_high = vals >= gp95
        count_high = int(mask_high.sum())
        earliest_high_ts = g.loc[mask_high, 'ts_dt'].min() if count_high>0 else pd.NaT
    else:
        count_high = 0
        earliest_high_ts = pd.NaT
    max_value_in_window = vals.max() if not g['value'].isna().all() else pd.NA

    rows.append({
        'cmdb_id': cmdb_id,
        'kpi_name': kpi_name,
        'global_P95': gp95,
        'count_high': count_high,
        'earliest_high_timestamp': earliest_high_ts,
        'max_value_in_window': max_value_in_window
    })

tomcat_disk_summary = pd.DataFrame(rows)

# 3b) Also prepare up to 50 raw points across both components sorted by timestamp ascending
# raw points columns: timestamp, cmdb_id, kpi_name, value
tomcat_disk_points = window_points[['ts_dt','cmdb_id','kpi_name','value']].rename(columns={'ts_dt':'timestamp'}).sort_values('timestamp', ascending=True).reset_index(drop=True)

# 4) Return compact tables: (A) per-series summary (top 20 rows), (B) raw points limited to 50 rows
# Ensure consistent ordering and types
if not tomcat_disk_summary.empty:
    tomcat_disk_summary = tomcat_disk_summary.sort_values(['cmdb_id','kpi_name']).reset_index(drop=True)

# Prepare displayed limited tables
tomcat_disk_summary_display = tomcat_disk_summary[['cmdb_id','kpi_name','global_P95','count_high','earliest_high_timestamp','max_value_in_window']].head(20)
tomcat_disk_points_display  = tomcat_disk_points[['timestamp','cmdb_id','kpi_name','value']].head(50)

# Display only the limited tables (full results stored in tomcat_disk_summary and tomcat_disk_points)
tomcat_disk_summary_display, tomcat_disk_points_display

```
Out[6]:
```
```python
# Summarize disk-related metric analysis for Tomcat01 and Tomcat03.
# Uses in-memory variables: tomcat_disk_summary, tomcat_disk_points

import pandas as pd
import numpy as np

# Safety checks
if 'tomcat_disk_summary' not in globals() or 'tomcat_disk_points' not in globals():
    summary = "Required results (tomcat_disk_summary, tomcat_disk_points) are not available in the kernel."
else:
    s = tomcat_disk_summary.copy()
    p = tomcat_disk_points.copy()

    # Basic counts
    n_series = int(s.shape[0])
    n_points = int(p.shape[0])

    # Time range of collected raw points
    if n_points > 0:
        t_start = p['timestamp'].min()
        t_end = p['timestamp'].max()
        t_start_s = t_start.isoformat() if hasattr(t_start, 'isoformat') else str(t_start)
        t_end_s = t_end.isoformat() if hasattr(t_end, 'isoformat') else str(t_end)
    else:
        t_start_s = t_end_s = "N/A"

    # Compute severity_high = max_value_in_window / global_P95 where applicable
    def safe_sev(row):
        gp95 = row['global_P95']
        mv = row['max_value_in_window']
        if pd.isna(gp95) or gp95 == 0 or pd.isna(mv):
            return np.nan
        return float(mv) / float(gp95)

    if n_series > 0:
        s['severity_high'] = s.apply(safe_sev, axis=1)
        # Top series by severity_high (descending), ignoring NaN
        top_by_sev = s.dropna(subset=['severity_high']).sort_values('severity_high', ascending=False).head(5)
        # Top by count_high as fallback
        top_by_count = s.sort_values('count_high', ascending=False).head(5)
    else:
        top_by_sev = s.head(0)
        top_by_count = s.head(0)

    # Build summary lines
    lines = []
    lines.append("Disk-metrics analysis summary for Tomcat01 and Tomcat03 (window 2021-03-24 11:00–11:30 UTC+8):")
    lines.append(f"- Discovered disk-related KPI series: {n_series}")
    lines.append(f"- Collected raw metric points in window: {n_points} (time range: {t_start_s} → {t_end_s})")

    if n_series == 0:
        lines.append("- No disk-related KPI series found for Tomcat01 or Tomcat03.")
    else:
        lines.append("- Top series by severity (max_in_window / global_P95):")
        if top_by_sev.shape[0] == 0:
            lines.append("  (no series with computable severity; some global_P95 values are 0 or missing)")
        else:
            for _, r in top_by_sev.iterrows():
                sev = r['severity_high']
                sev_s = f"{sev:.3f}" if not pd.isna(sev) else "NA"
                eh = r['earliest_high_timestamp']
                eh_s = eh.isoformat() if (not pd.isna(eh) and hasattr(eh, 'isoformat')) else "None"
                lines.append(
                    f"  • {r['cmdb_id']} / {r['kpi_name']}: severity={sev_s}, "
                    f"count_high={int(r['count_high'])}, earliest_high={eh_s}, max_in_window={r['max_value_in_window']}, global_P95={r['global_P95']}"
                )

        # Also highlight top by count_high
        lines.append("- Top series by count_high (number of points >= global_P95 in window):")
        for _, r in top_by_count.iterrows():
            eh = r['earliest_high_timestamp']
            eh_s = eh.isoformat() if (not pd.isna(eh) and hasattr(eh, 'isoformat')) else "None"
            lines.append(
                f"  • {r['cmdb_id']} / {r['kpi_name']}: count_high={int(r['count_high'])}, earliest_high={eh_s}, global_P95={r['global_P95']}, max_in_window={r['max_value_in_window']}"
            )

        # Notes about zero thresholds and zero-max cases
        lines.append("- Notes:")
        lines.append("  • Several series have global_P95 == 0; severity cannot be computed for those (treated as NA).")
        lines.append("  • Some series show many zero values in-window (e.g., DSKRead/DSKRTps) producing count_high where global_P95==0; interpret carefully.")
        lines.append("  • Raw points (up to 50) are available in variable 'tomcat_disk_points' for direct inspection.")

    summary = "\n".join(lines)

summary
```

The original code execution output of IPython Kernel is also provided below for reference:

(     cmdb_id                                           kpi_name  global_P95  count_high   earliest_high_timestamp  max_value_in_window
0   Tomcat01  OSLinux-OSLinux_LOCALDISK_LOCALDISK-sda_DSKAvg...    0.860000           6 2021-03-24 11:03:00+08:00             0.860000
1   Tomcat01     OSLinux-OSLinux_LOCALDISK_LOCALDISK-sda_DSKBps    0.300000           1 2021-03-24 11:07:00+08:00             1.000000
2   Tomcat01  OSLinux-OSLinux_LOCALDISK_LOCALDISK-sda_DSKPer...    0.300000           2 2021-03-24 11:17:00+08:00             0.300000
3   Tomcat01    OSLinux-OSLinux_LOCALDISK_LOCALDISK-sda_DSKRTps    0.183333           3 2021-03-24 11:01:00+08:00             0.183333
4   Tomcat01    OSLinux-OSLinux_LOCALDISK_LOCALDISK-sda_DSKRead    6.000000           3 2021-03-24 11:01:00+08:00             6.000000
5   Tomcat01  OSLinux-OSLinux_LOCALDISK_LOCALDISK-sda_DSKRea...   12.000000           2 2021-03-24 11:21:00+08:00            12.000000
6   Tomcat01     OSLinux-OSLinux_LOCALDISK_LOCALDISK-sda_DSKTps    0.000000           8 2021-03-24 11:03:00+08:00             0.000000
7   Tomcat01    OSLinux-OSLinux_LOCALDISK_LOCALDISK-sda_DSKWTps    0.824166           1 2021-03-24 11:11:00+08:00             0.900000
8   Tomcat01   OSLinux-OSLinux_LOCALDISK_LOCALDISK-sda_DSKWrite   12.500000           1 2021-03-24 11:11:00+08:00            12.933333
9   Tomcat01  OSLinux-OSLinux_LOCALDISK_LOCALDISK-sdb_DSKAvg...    0.860000           6 2021-03-24 11:03:00+08:00             0.860000
10  Tomcat01     OSLinux-OSLinux_LOCALDISK_LOCALDISK-sdb_DSKBps    1.700000           1 2021-03-24 11:23:00+08:00             3.000000
11  Tomcat01  OSLinux-OSLinux_LOCALDISK_LOCALDISK-sdb_DSKPer...    0.300000           3 2021-03-24 11:03:00+08:00             0.300000
12  Tomcat01    OSLinux-OSLinux_LOCALDISK_LOCALDISK-sdb_DSKRTps    0.000000          14 2021-03-24 11:01:00+08:00             0.000000
13  Tomcat01    OSLinux-OSLinux_LOCALDISK_LOCALDISK-sdb_DSKRead    0.000000          14 2021-03-24 11:01:00+08:00             0.000000
14  Tomcat01  OSLinux-OSLinux_LOCALDISK_LOCALDISK-sdb_DSKRea...  293.350000           1 2021-03-24 11:27:00+08:00           307.000000
15  Tomcat01     OSLinux-OSLinux_LOCALDISK_LOCALDISK-sdb_DSKTps    1.000000           5 2021-03-24 11:15:00+08:00             1.000000
16  Tomcat01    OSLinux-OSLinux_LOCALDISK_LOCALDISK-sdb_DSKWTps    1.392500           1 2021-03-24 11:05:00+08:00             1.533333
17  Tomcat01   OSLinux-OSLinux_LOCALDISK_LOCALDISK-sdb_DSKWrite  284.233333           1 2021-03-24 11:07:00+08:00           284.666667
18  Tomcat03  OSLinux-OSLinux_LOCALDISK_LOCALDISK-sda_DSKAvg...    0.880000           6 2021-03-24 11:03:00+08:00             0.880000
19  Tomcat03     OSLinux-OSLinux_LOCALDISK_LOCALDISK-sda_DSKBps    2.400000           1 2021-03-24 11:07:00+08:00             5.000000,                    timestamp   cmdb_id                                           kpi_name       value
0  2021-03-24 11:00:00+08:00  Tomcat01     OSLinux-OSLinux_LOCALDISK_LOCALDISK-sda_DSKBps    0.000000
1  2021-03-24 11:00:00+08:00  Tomcat01  OSLinux-OSLinux_LOCALDISK_LOCALDISK-sda_DSKPer...    0.000000
2  2021-03-24 11:01:00+08:00  Tomcat03   OSLinux-OSLinux_LOCALDISK_LOCALDISK-sdb_DSKWrite  240.266667
3  2021-03-24 11:01:00+08:00  Tomcat03    OSLinux-OSLinux_LOCALDISK_LOCALDISK-sdb_DSKWTps    1.166667
4  2021-03-24 11:01:00+08:00  Tomcat03     OSLinux-OSLinux_LOCALDISK_LOCALDISK-sdb_DSKTps    0.000000
..                       ...       ...                                                ...         ...
45 2021-03-24 11:03:00+08:00  Tomcat03     OSLinux-OSLinux_LOCALDISK_LOCALDISK-sda_DSKTps    1.000000
46 2021-03-24 11:03:00+08:00  Tomcat01   OSLinux-OSLinux_LOCALDISK_LOCALDISK-sdb_DSKWrite  240.800000
47 2021-03-24 11:03:00+08:00  Tomcat03  OSLinux-OSLinux_LOCALDISK_LOCALDISK-sda_DSKAvg...    0.880000
48 2021-03-24 11:03:00+08:00  Tomcat01  OSLinux-OSLinux_LOCALDISK_LOCALDISK-sda_DSKAvg...    0.860000
49 2021-03-24 11:03:00+08:00  Tomcat01    OSLinux-OSLinux_LOCALDISK_LOCALDISK-sda_DSKRead    0.000000

[50 rows x 4 columns])

**Note**: The printed pandas DataFrame is truncated due to its size. Only **10 rows** are displayed, which may introduce observation bias due to the incomplete table. If you want to comprehensively understand the details without bias, please ask Executor using `df.head(X)` to display more rows.```
```

In [ ]:
# In[7]:

import pandas as pd
import pytz
tz = pytz.timezone('Asia/Shanghai')

# Reuse existing in-memory variables: metric_anom_full, trace_anom_full, error_logs_window
# Safety checks - create empty DataFrames if missing
if 'metric_anom_full' not in globals() or metric_anom_full is None:
    metric_anom_full = pd.DataFrame(columns=[
        'cmdb_id','kpi_name','global_P95','global_P5','count_high','earliest_high_timestamp',
        'max_value_in_window','severity_high','count_low','earliest_low_timestamp',
        'min_value_in_window','severity_low'
    ])
if 'trace_anom_full' not in globals() or trace_anom_full is None:
    trace_anom_full = pd.DataFrame(columns=[
        'cmdb_id','trace_name','global_P95','global_P5','count_high','earliest_high_timestamp',
        'max_value_in_window','severity_high','count_low','earliest_low_timestamp',
        'min_value_in_window','severity_low'
    ])
if 'error_logs_window' not in globals() or error_logs_window is None:
    error_logs_window = pd.DataFrame(columns=['timestamp','cmdb_id','message'])

# Ensure severity numeric helper columns exist (compute if missing)
def ensure_severity_cols(df):
    # create numeric severity_high_num, severity_low_num, severity_max
    if 'severity_high_num' not in df.columns:
        df['severity_high_num'] = df['severity_high'].apply(lambda x: 0.0 if (pd.isna(x) or x is pd.NA) else float(x))
    if 'severity_low_num' not in df.columns:
        df['severity_low_num'] = df['severity_low'].apply(lambda x: 0.0 if (pd.isna(x) or x is pd.NA) else float(x))
    df['severity_max'] = df[['severity_high_num','severity_low_num']].max(axis=1)
    return df

metric_anom_full = ensure_severity_cols(metric_anom_full.copy())
trace_anom_full = ensure_severity_cols(trace_anom_full.copy())

# 1) Filter metric rows with severity_max >= 1.5
metric_selected = metric_anom_full[metric_anom_full['severity_max'] >= 1.5].copy()
# Prepare unified columns for metric rows
if not metric_selected.empty:
    # Decide which earliest timestamp to use per row (choose the side that produced severity_max)
    def pick_earliest(row):
        # prefer high if equal or greater
        if row['severity_high_num'] >= row['severity_low_num']:
            return row.get('earliest_high_timestamp', pd.NaT)
        else:
            return row.get('earliest_low_timestamp', pd.NaT)
    metric_selected['earliest_timestamp_chosen'] = metric_selected.apply(pick_earliest, axis=1)
    metric_unified = pd.DataFrame({
        'source': 'metric',
        'cmdb_id': metric_selected['cmdb_id'],
        'series_name': metric_selected['kpi_name'],
        'severity_max': metric_selected['severity_max'],
        'earliest_high_timestamp': metric_selected['earliest_timestamp_chosen'],
        'count_high': metric_selected['count_high'],
        'max_value_in_window': metric_selected['max_value_in_window']
    })
else:
    metric_unified = pd.DataFrame(columns=['source','cmdb_id','series_name','severity_max','earliest_high_timestamp','count_high','max_value_in_window'])

# 2) Filter trace rows with severity_max >= 1.10
trace_selected = trace_anom_full[trace_anom_full['severity_max'] >= 1.10].copy()
if not trace_selected.empty:
    def pick_earliest_trace(row):
        if row['severity_high_num'] >= row['severity_low_num']:
            return row.get('earliest_high_timestamp', pd.NaT)
        else:
            return row.get('earliest_low_timestamp', pd.NaT)
    trace_selected['earliest_timestamp_chosen'] = trace_selected.apply(pick_earliest_trace, axis=1)
    trace_unified = pd.DataFrame({
        'source': 'trace',
        'cmdb_id': trace_selected['cmdb_id'],
        'series_name': trace_selected['trace_name'],
        'severity_max': trace_selected['severity_max'],
        'earliest_high_timestamp': trace_selected['earliest_timestamp_chosen'],
        'count_high': trace_selected['count_high'],
        'max_value_in_window': trace_selected['max_value_in_window']
    })
else:
    trace_unified = pd.DataFrame(columns=['source','cmdb_id','series_name','severity_max','earliest_high_timestamp','count_high','max_value_in_window'])

# 3) From error_logs_window compute per-cmdb_id earliest error timestamp and count
if not error_logs_window.empty:
    # ensure timestamp column is tz-aware datetime (should be)
    err_grp = error_logs_window.groupby('cmdb_id', sort=False)
    err_agg = err_grp.agg(
        count_errors=('message','count'),
        earliest_error=('timestamp','min')
    ).reset_index()
    error_unified = pd.DataFrame({
        'source': 'error_log',
        'cmdb_id': err_agg['cmdb_id'],
        'series_name': ['error_messages'] * len(err_agg),
        'severity_max': [pd.NA] * len(err_agg),
        'earliest_high_timestamp': err_agg['earliest_error'],
        'count_high': err_agg['count_errors'],
        'max_value_in_window': [pd.NA] * len(err_agg)
    })
else:
    error_unified = pd.DataFrame(columns=['source','cmdb_id','series_name','severity_max','earliest_high_timestamp','count_high','max_value_in_window'])

# 4) Build the unified candidate_faults_full table
candidate_faults_full = pd.concat([metric_unified, trace_unified, error_unified], ignore_index=True, sort=False)

# Ensure earliest_high_timestamp is datetime tz-aware; convert if necessary
if 'earliest_high_timestamp' in candidate_faults_full.columns:
    # convert strings or objects to datetime if needed
    candidate_faults_full['earliest_high_timestamp'] = pd.to_datetime(candidate_faults_full['earliest_high_timestamp'], utc=True, errors='coerce').dt.tz_convert(tz)

# 5) Return limited top 20 rows sorted by earliest_high_timestamp ascending
# Rows with NaT will appear last
candidate_faults_full = candidate_faults_full.sort_values('earliest_high_timestamp', ascending=True).reset_index(drop=True)

candidate_faults_top = candidate_faults_full.head(20)[['source','cmdb_id','series_name','severity_max','earliest_high_timestamp','count_high','max_value_in_window']]

# One-line summary listing the single earliest row (component and timestamp)
earliest_row = candidate_faults_full.loc[candidate_faults_full['earliest_high_timestamp'].first_valid_index()] if candidate_faults_full['earliest_high_timestamp'].first_valid_index() is not None else None

if earliest_row is not None and not earliest_row.empty:
    earliest_summary = f"Earliest candidate fault: {earliest_row['cmdb_id']} at {earliest_row['earliest_high_timestamp'].isoformat()} (source={earliest_row['source']}, series={earliest_row['series_name']})"
else:
    # fallback: if no valid earliest timestamp, pick first row if exists
    if not candidate_faults_full.empty:
        r = candidate_faults_full.iloc[0]
        ts = r['earliest_high_timestamp']
        ts_s = ts.isoformat() if not pd.isna(ts) and hasattr(ts, 'isoformat') else "None"
        earliest_summary = f"Earliest candidate fault (fallback): {r['cmdb_id']} at {ts_s} (source={r['source']}, series={r['series_name']})"
    else:
        earliest_summary = "No candidate faults found."

# Store full table in kernel variable candidate_faults_full (already assigned)
candidate_faults_top, earliest_summary

```
Out[7]:
```
```python
# Build a concise plain-English summary of the candidate faults table using in-kernel results.
# Uses variables: candidate_faults_top, earliest_summary (from previous step)

import pandas as pd

# Safety checks
if 'candidate_faults_top' not in globals() or 'earliest_summary' not in globals():
    summary = "Candidate faults table not available in the kernel."
else:
    df = candidate_faults_top.copy()
    # Top components by severity_max (non-null), up to 5
    df_sev = df[df['severity_max'].notna()].sort_values('severity_max', ascending=False).head(5)
    top_list = []
    for _, r in df_sev.iterrows():
        sev = float(r['severity_max']) if pd.notna(r['severity_max']) else None
        sev_s = f"{sev:.3f}" if sev is not None else "NA"
        ts = r['earliest_high_timestamp']
        ts_s = ts.isoformat() if (not pd.isna(ts) and hasattr(ts, 'isoformat')) else "None"
        top_list.append(f"{r['cmdb_id']} ({r['series_name']}, severity={sev_s}, time={ts_s})")

    # Compose summary lines
    lines = []
    lines.append("Candidate faults summary (compact):")
    lines.append(f"- Earliest candidate fault: {earliest_summary}")
    lines.append(f"- Total candidate rows displayed: {len(df)} (combined metric/trace/error sources).")
    if top_list:
        lines.append("- Top anomalous series by severity (up to 5):")
        for item in top_list:
            lines.append(f"  • {item}")
    else:
        lines.append("- No entries with computable severity found in the displayed rows.")

    lines.append("- Observations: disk-related anomalies (e.g., Redis02 DSKWrite, IG01 DSKTps) and CPU/JVM spikes (Tomcat03, IG01) appear among top candidates;")
    lines.append("  trace anomalies concentrate around MG02 and Tomcat→MG interactions, indicating increased inter-service row_count/duration metrics.")
    lines.append("  Use the full table 'candidate_faults_full' for detailed follow-up.")

    summary = "\n".join(lines)

summary
```

The original code execution output of IPython Kernel is also provided below for reference:

(    source   cmdb_id                                        series_name  severity_max   earliest_high_timestamp  count_high  max_value_in_window
0   metric   Redis02   OSLinux-OSLinux_LOCALDISK_LOCALDISK-sda_DSKWrite      1.574803 2021-03-24 11:00:00+08:00           1         1.866667e+01
1   metric      IG01     OSLinux-OSLinux_LOCALDISK_LOCALDISK-sda_DSKTps      1.714286 2021-03-24 11:00:00+08:00           1         6.000000e+00
2    trace      MG02                  trace.from_dockerB1.duration_mean      1.569420 2021-03-24 11:00:00+08:00           2         2.697660e-01
3    trace      MG02                  trace.from_dockerB2.duration_mean      1.340814 2021-03-24 11:00:00+08:00           2         2.751430e-01
4    trace      MG02                     trace.to_dockerB2.duration_p95      1.502450 2021-03-24 11:00:00+08:00           2         5.305900e+00
5   metric  Tomcat03                    OSLinux-CPU_CPU-2_SingleCpuidle      6.039254 2021-03-24 11:01:00+08:00           1         9.875229e+01
6   metric   Mysql02  OSLinux-OSLinux_LOCALDISK_LOCALDISK-sda_DSKPer...      1.538462 2021-03-24 11:01:00+08:00           1         1.600000e+00
7   metric  Tomcat03                    OSLinux-CPU_CPU-2_SingleCpuUtil      1.511991 2021-03-24 11:01:00+08:00           2         1.000000e+02
8    trace      MG02                      trace.from_dockerB2.row_count     16.250000 2021-03-24 11:01:00+08:00           2         1.340000e+02
9    trace      IG01                            trace.self.duration_p95      1.167257 2021-03-24 11:01:00+08:00           2         6.101600e+01
10   trace      IG01                           trace.self.duration_mean      1.240456 2021-03-24 11:01:00+08:00           2         7.135872e+00
11  metric   Redis02     OSLinux-OSLinux_LOCALDISK_LOCALDISK-sda_DSKTps      1.666667 2021-03-24 11:02:00+08:00           1         1.000000e+00
12  metric      IG01          JVM-Memory_7778_JVM_Memory_HeapMemoryUsed      1.647729 2021-03-24 11:03:00+08:00           2         8.490422e+08
13   trace      MG02                      trace.from_dockerA2.row_count      1.530769 2021-03-24 11:03:00+08:00           2         1.350000e+02
14   trace      MG02                  trace.from_dockerA1.duration_mean      4.605176 2021-03-24 11:03:00+08:00           2         1.331750e+00
15   trace  Tomcat01                         trace.to_MG02.duration_p95     35.546531 2021-03-24 11:03:00+08:00           2         3.863600e+00
16   trace      MG02                   trace.from_Tomcat01.duration_p95     35.546531 2021-03-24 11:03:00+08:00           2         3.863600e+00
17  metric   Redis02                            OSLinux-CPU_CPU_CPULoad      1.502591 2021-03-24 11:03:00+08:00           2         2.900000e-01
18  metric   Mysql02          Mysql-MySQL_3306_CurrentSQLMaxRunningTime      1.666667 2021-03-24 11:04:00+08:00           2         1.000000e+00
19  metric   Mysql02              Mysql-MySQL_3306_LongestTrxActiveTime      1.666667 2021-03-24 11:04:00+08:00           2         1.000000e+00, 'Earliest candidate fault: Redis02 at 2021-03-24T11:00:00+08:00 (source=metric, series=OSLinux-OSLinux_LOCALDISK_LOCALDISK-sda_DSKWrite)')```
```

In [ ]:
# In[8]:

import pandas as pd
import pytz
tz = pytz.timezone('Asia/Shanghai')

# Reuse trace_anom_full from kernel
try:
    trace_anom_full
except NameError:
    trace_anom_full = pd.DataFrame()  # safety fallback

# Candidate target components
targets = ['Redis02','IG01','MG02','Tomcat01','Tomcat02','Tomcat03']

# Ensure expected columns exist
required_cols = {'cmdb_id','trace_name','count_high','earliest_high_timestamp','max_value_in_window','severity_high'}
missing = required_cols - set(trace_anom_full.columns)
if missing:
    # Build empty result if trace_anom_full lacks required data
    trace_to_targets = pd.DataFrame(columns=['target_component','cmdb_id','trace_name','earliest_high_timestamp','max_value_in_window','severity_high'])
else:
    df = trace_anom_full.copy()

    # Ensure earliest_high_timestamp is datetime tz-aware in UTC+8
    if not pd.api.types.is_datetime64_any_dtype(df['earliest_high_timestamp']):
        df['earliest_high_timestamp'] = pd.to_datetime(df['earliest_high_timestamp'], utc=True, errors='coerce').dt.tz_convert(tz)
    else:
        # if dtype is datetime64[ns, tz], convert to Asia/Shanghai to be consistent
        try:
            df['earliest_high_timestamp'] = df['earliest_high_timestamp'].dt.tz_convert(tz)
        except Exception:
            df['earliest_high_timestamp'] = pd.to_datetime(df['earliest_high_timestamp'], utc=True, errors='coerce').dt.tz_convert(tz)

    # Filter rows with count_high > 0
    df = df[df['count_high'].fillna(0) > 0].copy()

    # Find rows where trace_name contains 'to_<target>' for any target in list
    # We'll search for substring 'to_<Target>' (case-sensitive) within trace_name
    matches = []
    for target in targets:
        mask = df['trace_name'].str.contains(f"to_{target}", na=False)
        if mask.any():
            sub = df[mask].copy()
            sub = sub.assign(target_component=target)
            matches.append(sub)

    if matches:
        trace_to_targets_full = pd.concat(matches, ignore_index=True, sort=False)
        # Select required columns and rename target_component
        trace_to_targets = trace_to_targets_full[['target_component','cmdb_id','trace_name','earliest_high_timestamp','max_value_in_window','severity_high']].copy()
        # Sort by earliest_high_timestamp ascending and limit to top 50
        trace_to_targets = trace_to_targets.sort_values('earliest_high_timestamp', ascending=True).reset_index(drop=True)
    else:
        trace_to_targets = pd.DataFrame(columns=['target_component','cmdb_id','trace_name','earliest_high_timestamp','max_value_in_window','severity_high'])

# Store full result in kernel variable 'trace_to_targets' (already assigned)
# Display only limited table (top 50 rows) as requested
trace_to_targets_head = trace_to_targets.head(50)
trace_to_targets_head

```
Out[8]:
```
```python
# Summarize trace->target anomalies found in trace_to_targets (compact plain English)
import pandas as pd

# Safety: use trace_to_targets if present
df = trace_to_targets.head(1000).copy() if 'trace_to_targets' in globals() else pd.DataFrame()

if df.empty:
    summary = "No trace->target anomaly rows found for the requested targets in the failure window."
else:
    # Ensure datetime dtype
    if not pd.api.types.is_datetime64_any_dtype(df['earliest_high_timestamp']):
        df['earliest_high_timestamp'] = pd.to_datetime(df['earliest_high_timestamp'], utc=True, errors='coerce').dt.tz_convert('Asia/Shanghai')

    # Which target components and counts
    target_counts = df['target_component'].value_counts().to_dict()

    # Earliest anomaly overall
    earliest_idx = df['earliest_high_timestamp'].idxmin()
    earliest_row = df.loc[earliest_idx]
    earliest_comp = earliest_row['target_component']
    earliest_owner = earliest_row['cmdb_id']
    earliest_ts = earliest_row['earliest_high_timestamp'].isoformat()
    earliest_trace = earliest_row['trace_name']

    # Earliest per target (first timestamp where that target was hit)
    earliest_per_target = df.groupby('target_component', sort=False)['earliest_high_timestamp'].min().to_dict()
    earliest_per_target_str = {k: v.isoformat() for k,v in earliest_per_target.items()}

    # Top 3 events by severity_high
    top_sev = df.sort_values('severity_high', ascending=False).head(3)
    top_sev_lines = []
    for _, r in top_sev.iterrows():
        ts = r['earliest_high_timestamp'].isoformat() if not pd.isna(r['earliest_high_timestamp']) else "None"
        top_sev_lines.append(f"{r['target_component']} targeted by {r['cmdb_id']} / {r['trace_name']}: severity={float(r['severity_high']):.3f}, time={ts}, max={r['max_value_in_window']}")

    # Build summary text
    lines = []
    lines.append("Trace->target anomaly summary (window 2021-03-24 11:00–11:30 UTC+8):")
    lines.append(f"- Targets observed: {', '.join(sorted(target_counts.keys()))}")
    lines.append("- Counts per target (number of trace-series rows with count_high>0):")
    for t, c in target_counts.items():
        lines.append(f"  • {t}: {c} row(s), earliest at {earliest_per_target_str.get(t)}")
    lines.append(f"- Earliest observed target-directed anomaly: {earliest_comp} was targeted by {earliest_owner} at {earliest_ts} (trace: {earliest_trace})")
    lines.append("- Top 3 highest-severity target-directed trace events:")
    for l in top_sev_lines:
        lines.append(f"  • {l}")
    lines.append("- Note: entries include traces like 'to_MG02', 'to_Tomcat01/02/03' showing spikes (severity typically ~1.05–3.22).")

    summary = "\n".join(lines)

summary
```

The original code execution output of IPython Kernel is also provided below for reference:

target_component   cmdb_id                       trace_name   earliest_high_timestamp  max_value_in_window  severity_high
0              MG02  Tomcat01          trace.to_MG02.row_count 2021-03-24 11:00:00+08:00           329.000000       1.073409
1          Tomcat01      IG01      trace.to_Tomcat01.row_count 2021-03-24 11:00:00+08:00           337.000000       1.056426
2              MG02  Tomcat02          trace.to_MG02.row_count 2021-03-24 11:01:00+08:00           357.000000       1.122642
3          Tomcat03      IG01      trace.to_Tomcat03.row_count 2021-03-24 11:04:00+08:00           338.000000       1.113674
4          Tomcat02      IG01      trace.to_Tomcat02.row_count 2021-03-24 11:05:00+08:00           327.000000       1.033175
5              MG02  Tomcat02       trace.to_MG02.duration_p95 2021-03-24 11:06:00+08:00             3.472800       1.124757
6              MG02  Tomcat03          trace.to_MG02.row_count 2021-03-24 11:07:00+08:00           331.000000       1.092409
7          Tomcat02      IG01   trace.to_Tomcat02.duration_p95 2021-03-24 11:12:00+08:00             4.339600       1.077218
8          Tomcat02      IG01  trace.to_Tomcat02.duration_mean 2021-03-24 11:13:00+08:00             1.017910       1.057315
9          Tomcat01      IG01  trace.to_Tomcat01.duration_mean 2021-03-24 11:13:00+08:00             1.068717       1.210349
10         Tomcat03      IG01  trace.to_Tomcat03.duration_mean 2021-03-24 11:13:00+08:00             1.037943       1.207188
11         Tomcat03      IG01   trace.to_Tomcat03.duration_p95 2021-03-24 11:13:00+08:00             4.931000       1.265811
12             MG02  Tomcat02      trace.to_MG02.duration_mean 2021-03-24 11:13:00+08:00             0.669502       1.384785
13         Tomcat01      IG01   trace.to_Tomcat01.duration_p95 2021-03-24 11:13:00+08:00             5.096000       1.226740
14             MG02  Tomcat01      trace.to_MG02.duration_mean 2021-03-24 11:14:00+08:00             0.537862       1.050899
15             MG02  Tomcat01       trace.to_MG02.duration_p95 2021-03-24 11:14:00+08:00             3.863600       1.191228
16             MG02  Tomcat03      trace.to_MG02.duration_mean 2021-03-24 11:16:00+08:00             1.536500       3.221066
17             MG02  Tomcat03       trace.to_MG02.duration_p95 2021-03-24 11:20:00+08:00             6.742750       2.196836```
```

In [ ]:
# In[9]:

import pandas as pd
import pytz

tz = pytz.timezone('Asia/Shanghai')

# Candidate components
components = ['Redis02','IG01','MG02','Tomcat01','Tomcat02','Tomcat03','apache01','apache02']

# Ensure source DataFrames exist
if 'metric_anom_full' not in globals():
    metric_anom_full = pd.DataFrame()
if 'trace_to_targets' not in globals():
    trace_to_targets = pd.DataFrame()
if 'trace_anom_full' not in globals():
    trace_anom_full = pd.DataFrame()
if 'error_logs_window' not in globals():
    error_logs_window = pd.DataFrame()

# Normalize datetime columns to tz-aware Asia/Shanghai where present
def ensure_dt(col):
    if col.name in (metric_anom_full.columns if isinstance(metric_anom_full, pd.DataFrame) else []):
        pass
# Ensure metric_anom_full timestamp cols
for col in ['earliest_high_timestamp','earliest_low_timestamp']:
    if col in metric_anom_full.columns:
        metric_anom_full[col] = pd.to_datetime(metric_anom_full[col], utc=True, errors='coerce').dt.tz_convert(tz)

# Ensure trace_to_targets earliest_high_timestamp
if 'earliest_high_timestamp' in trace_to_targets.columns:
    trace_to_targets['earliest_high_timestamp'] = pd.to_datetime(trace_to_targets['earliest_high_timestamp'], utc=True, errors='coerce').dt.tz_convert(tz)

# Ensure trace_anom_full earliest timestamps
for col in ['earliest_high_timestamp','earliest_low_timestamp']:
    if col in trace_anom_full.columns:
        trace_anom_full[col] = pd.to_datetime(trace_anom_full[col], utc=True, errors='coerce').dt.tz_convert(tz)

# Ensure error_logs_window timestamp column is tz-aware and named 'timestamp' (it should be)
if 'timestamp' in error_logs_window.columns:
    error_logs_window['timestamp'] = pd.to_datetime(error_logs_window['timestamp'], utc=True, errors='coerce').dt.tz_convert(tz)

# Helper to get min timestamp from a Series (ignores NaT)
def min_ts(series):
    if series.dropna().empty:
        return pd.NaT
    return series.dropna().min()

rows = []
for comp in components:
    # 1) Metric fault info
    if not metric_anom_full.empty:
        mrows = metric_anom_full[metric_anom_full['cmdb_id'] == comp].copy()
        # consider rows where severity_max is defined (not NA)
        mrows_valid = mrows[mrows['severity_max'].notna()]
        if not mrows_valid.empty:
            has_metric_fault = True
            # take smallest earliest_high_timestamp among those rows (dropna)
            metric_earliest = min_ts(mrows_valid['earliest_high_timestamp'])
            metric_max_severity = float(mrows_valid['severity_max'].max())
        else:
            has_metric_fault = False
            metric_earliest = pd.NaT
            metric_max_severity = pd.NA
    else:
        has_metric_fault = False
        metric_earliest = pd.NaT
        metric_max_severity = pd.NA

    # 2) Incoming-trace target info: prefer trace_to_targets, fallback to trace_anom_full search
    incoming_df = pd.DataFrame()
    if not trace_to_targets.empty:
        incoming_df = trace_to_targets[trace_to_targets['target_component'] == comp].copy()
    # fallback
    if incoming_df.empty and not trace_anom_full.empty:
        # find rows in trace_anom_full where trace_name contains 'to_<comp>' and count_high>0
        mask = trace_anom_full['trace_name'].str.contains(f"to_{comp}", na=False) & (trace_anom_full['count_high'].fillna(0) > 0)
        incoming_df = trace_anom_full[mask].copy()
        # standardize column names
        if not incoming_df.empty and 'severity_high' in incoming_df.columns:
            # keep as is; ensure earliest_high_timestamp present and tz-aware
            incoming_df['earliest_high_timestamp'] = pd.to_datetime(incoming_df['earliest_high_timestamp'], utc=True, errors='coerce').dt.tz_convert(tz)

    if not incoming_df.empty:
        has_incoming_trace = True
        trace_incoming_earliest = min_ts(incoming_df['earliest_high_timestamp'])
        # incoming_trace_max_severity = max of severity_high where defined
        if 'severity_high' in incoming_df.columns:
            incoming_vals = incoming_df['severity_high'].dropna().astype(float)
            incoming_trace_max_severity = float(incoming_vals.max()) if not incoming_vals.empty else pd.NA
        else:
            incoming_trace_max_severity = pd.NA
    else:
        has_incoming_trace = False
        trace_incoming_earliest = pd.NaT
        incoming_trace_max_severity = pd.NA

    # 3) Error-log info
    if not error_logs_window.empty:
        erows = error_logs_window[error_logs_window['cmdb_id'] == comp].copy()
        if not erows.empty:
            has_error_logs = True
            error_earliest = min_ts(erows['timestamp'])
            error_count = int(erows.shape[0])
        else:
            has_error_logs = False
            error_earliest = pd.NaT
            error_count = 0
    else:
        has_error_logs = False
        error_earliest = pd.NaT
        error_count = 0

    rows.append({
        'component': comp,
        'has_metric_fault': has_metric_fault,
        'metric_earliest': metric_earliest,
        'metric_max_severity': metric_max_severity,
        'has_incoming_trace': has_incoming_trace,
        'trace_incoming_earliest': trace_incoming_earliest,
        'incoming_trace_max_severity': incoming_trace_max_severity,
        'has_error_logs': has_error_logs,
        'error_earliest': error_earliest,
        'error_count': error_count
    })

candidate_correlation = pd.DataFrame(rows)

# Compute a first_event column as the earliest non-null among metric_earliest, trace_incoming_earliest, error_earliest
def first_event_ts(r):
    ts_candidates = [r['metric_earliest'], r['trace_incoming_earliest'], r['error_earliest']]
    # filter out NaT
    valid = [t for t in ts_candidates if pd.notna(t)]
    if not valid:
        return pd.NaT
    return min(valid)

candidate_correlation['first_event'] = candidate_correlation.apply(first_event_ts, axis=1)

# Sort by first_event ascending, NaT go last
candidate_correlation = candidate_correlation.sort_values('first_event', ascending=True).reset_index(drop=True)

# Drop helper column for display if desired, but keep in full variable. Display requested columns.
display_cols = ['component','has_metric_fault','metric_earliest','metric_max_severity',
                'has_incoming_trace','trace_incoming_earliest','incoming_trace_max_severity',
                'has_error_logs','error_earliest','error_count']

candidate_correlation = candidate_correlation[display_cols + ['first_event']]

# Keep full result in kernel variable 'candidate_correlation' and display the table (at most 8 rows)
candidate_correlation.head(8)

```
Out[9]:
```
```python
# Build a plain-English summary string from the candidate_correlation table and display it.
import pandas as pd

# Use existing candidate_correlation variable from kernel
df = candidate_correlation.copy() if 'candidate_correlation' in globals() else pd.DataFrame()

def fmt_ts(t):
    if pd.isna(t):
        return "None"
    try:
        return pd.to_datetime(t).tz_convert('Asia/Shanghai').isoformat()
    except Exception:
        return str(t)

def fmt_num(x):
    if x is pd.NA or pd.isna(x):
        return "None"
    try:
        return f"{float(x):.3f}"
    except Exception:
        return str(x)

if df.empty:
    summary = "No correlation data available."
else:
    lines = []
    lines.append("Candidate correlation summary (per component):")
    for _, r in df.iterrows():
        comp = r['component']
        # Metric info
        has_m = bool(r['has_metric_fault'])
        m_ear = fmt_ts(r['metric_earliest']) if has_m else "None"
        m_sev = fmt_num(r['metric_max_severity']) if has_m else "None"
        # Incoming trace info
        has_t = bool(r['has_incoming_trace'])
        t_ear = fmt_ts(r['trace_incoming_earliest']) if has_t else "None"
        t_sev = fmt_num(r['incoming_trace_max_severity']) if has_t else "None"
        # Error log info
        has_e = bool(r['has_error_logs'])
        e_ear = fmt_ts(r['error_earliest']) if has_e else "None"
        e_cnt = int(r['error_count']) if has_e and not pd.isna(r['error_count']) else 0

        lines.append(
            f"- {comp}: metric_fault={has_m} (earliest={m_ear}, max_sev={m_sev}); "
            f"incoming_trace={has_t} (earliest={t_ear}, max_sev={t_sev}); "
            f"error_logs={has_e} (earliest={e_ear}, count={e_cnt})"
        )

    summary = "\n".join(lines)

summary
```

The original code execution output of IPython Kernel is also provided below for reference:

component  has_metric_fault           metric_earliest  metric_max_severity  has_incoming_trace   trace_incoming_earliest incoming_trace_max_severity  has_error_logs            error_earliest  error_count               first_event
0   Redis02              True 2021-03-24 11:00:00+08:00             2.105263               False                       NaT                        <NA>           False                       NaT            0 2021-03-24 11:00:00+08:00
1      IG01              True 2021-03-24 11:00:00+08:00             1.877952               False                       NaT                        <NA>           False                       NaT            0 2021-03-24 11:00:00+08:00
2      MG02              True 2021-03-24 11:00:00+08:00             2.857143                True 2021-03-24 11:00:00+08:00                    3.221066           False                       NaT            0 2021-03-24 11:00:00+08:00
3  Tomcat01              True 2021-03-24 11:00:00+08:00             3.333333                True 2021-03-24 11:00:00+08:00                     1.22674           False                       NaT            0 2021-03-24 11:00:00+08:00
4  Tomcat02              True 2021-03-24 11:00:00+08:00             2.857143                True 2021-03-24 11:05:00+08:00                    1.077218           False                       NaT            0 2021-03-24 11:00:00+08:00
5  Tomcat03              True 2021-03-24 11:00:00+08:00             6.039254                True 2021-03-24 11:04:00+08:00                    1.265811           False                       NaT            0 2021-03-24 11:00:00+08:00
6  apache01              True 2021-03-24 11:00:00+08:00             1.937648               False                       NaT                        <NA>            True 2021-03-24 11:08:04+08:00          136 2021-03-24 11:00:00+08:00
7  apache02              True 2021-03-24 11:00:00+08:00             2.689759               False                       NaT                        <NA>            True 2021-03-24 11:08:00+08:00          151 2021-03-24 11:00:00+08:00```
```

In [ ]:
# In[10]:

import pandas as pd
import pytz

tz = pytz.timezone('Asia/Shanghai')

# Reuse existing kernel variables where available
# Ensure metric_anom_full, trace_to_targets, trace_anom_full, error_logs_window exist
if 'metric_anom_full' not in globals():
    metric_anom_full = pd.DataFrame()
if 'trace_to_targets' not in globals():
    trace_to_targets = pd.DataFrame()
if 'trace_anom_full' not in globals():
    trace_anom_full = pd.DataFrame()
if 'error_logs_window' not in globals():
    error_logs_window = pd.DataFrame()

# Normalize datetime columns to tz-aware Asia/Shanghai where present
if 'earliest_high_timestamp' in metric_anom_full.columns:
    metric_anom_full['earliest_high_timestamp'] = pd.to_datetime(metric_anom_full['earliest_high_timestamp'], utc=True, errors='coerce').dt.tz_convert(tz)
if 'earliest_low_timestamp' in metric_anom_full.columns:
    metric_anom_full['earliest_low_timestamp'] = pd.to_datetime(metric_anom_full['earliest_low_timestamp'], utc=True, errors='coerce').dt.tz_convert(tz)

if 'earliest_high_timestamp' in trace_to_targets.columns:
    trace_to_targets['earliest_high_timestamp'] = pd.to_datetime(trace_to_targets['earliest_high_timestamp'], utc=True, errors='coerce').dt.tz_convert(tz)
if 'earliest_high_timestamp' in trace_anom_full.columns:
    trace_anom_full['earliest_high_timestamp'] = pd.to_datetime(trace_anom_full['earliest_high_timestamp'], utc=True, errors='coerce').dt.tz_convert(tz)

if 'timestamp' in error_logs_window.columns:
    error_logs_window['timestamp'] = pd.to_datetime(error_logs_window['timestamp'], utc=True, errors='coerce').dt.tz_convert(tz)

# 1) Metric evidence for Tomcat01
if not metric_anom_full.empty:
    # Ensure severity_max exists; compute if missing
    if 'severity_max' not in metric_anom_full.columns:
        # create numeric severity helpers if available
        if 'severity_high' in metric_anom_full.columns:
            metric_anom_full['severity_high_num'] = metric_anom_full['severity_high'].apply(lambda x: 0.0 if (pd.isna(x) or x is pd.NA) else float(x))
        else:
            metric_anom_full['severity_high_num'] = 0.0
        if 'severity_low' in metric_anom_full.columns:
            metric_anom_full['severity_low_num'] = metric_anom_full['severity_low'].apply(lambda x: 0.0 if (pd.isna(x) or x is pd.NA) else float(x))
        else:
            metric_anom_full['severity_low_num'] = 0.0
        metric_anom_full['severity_max'] = metric_anom_full[['severity_high_num','severity_low_num']].max(axis=1)
    tomcat01_metric_evidence = metric_anom_full[metric_anom_full['cmdb_id'] == 'Tomcat01'].copy()
    # Select and sort by severity_max desc, limit top 10
    if not tomcat01_metric_evidence.empty:
        tomcat01_metric_evidence = tomcat01_metric_evidence.sort_values('severity_max', ascending=False).reset_index(drop=True)
    else:
        # ensure consistent columns if empty
        tomcat01_metric_evidence = pd.DataFrame(columns=['kpi_name','severity_max','earliest_high_timestamp','count_high','max_value_in_window'])
    # Prepare display table (up to 10 rows) with requested columns; kpi_name column may be named 'kpi_name'
    cols_metric = []
    if 'kpi_name' in tomcat01_metric_evidence.columns:
        cols_metric.append('kpi_name')
    else:
        # fallback to series name if different
        if 'series_name' in tomcat01_metric_evidence.columns:
            tomcat01_metric_evidence = tomcat01_metric_evidence.rename(columns={'series_name':'kpi_name'})
            cols_metric.append('kpi_name')
        else:
            # create empty kpi_name
            tomcat01_metric_evidence['kpi_name'] = pd.NA
            cols_metric.append('kpi_name')
    for c in ['severity_max','earliest_high_timestamp','count_high','max_value_in_window']:
        if c in tomcat01_metric_evidence.columns:
            cols_metric.append(c)
        else:
            tomcat01_metric_evidence[c] = pd.NA
            cols_metric.append(c)
    tomcat01_metric_evidence_display = tomcat01_metric_evidence[cols_metric].head(10)
else:
    tomcat01_metric_evidence = pd.DataFrame(columns=['kpi_name','severity_max','earliest_high_timestamp','count_high','max_value_in_window'])
    tomcat01_metric_evidence_display = tomcat01_metric_evidence.copy()

# 2) Trace evidence targeting Tomcat01
# Prefer trace_to_targets if available
if ('trace_to_targets' in globals()) and (not trace_to_targets.empty):
    tomcat01_trace_evidence = trace_to_targets[trace_to_targets['target_component'] == 'Tomcat01'].copy()
    # Ensure columns exist
    for col in ['cmdb_id','trace_name','earliest_high_timestamp','max_value_in_window','severity_high']:
        if col not in tomcat01_trace_evidence.columns:
            tomcat01_trace_evidence[col] = pd.NA
else:
    # Fallback to trace_anom_full searching trace_name containing to_Tomcat01 and count_high>0
    if not trace_anom_full.empty:
        mask = trace_anom_full['trace_name'].str.contains('to_Tomcat01', na=False) & (trace_anom_full['count_high'].fillna(0) > 0)
        tomcat01_trace_evidence = trace_anom_full[mask].copy()
        # Ensure expected columns
        for col in ['cmdb_id','trace_name','earliest_high_timestamp','max_value_in_window','severity_high']:
            if col not in tomcat01_trace_evidence.columns:
                tomcat01_trace_evidence[col] = pd.NA
    else:
        tomcat01_trace_evidence = pd.DataFrame(columns=['cmdb_id','trace_name','earliest_high_timestamp','max_value_in_window','severity_high'])

# Sort by earliest_high_timestamp ascending and limit to 20 rows for display
if not tomcat01_trace_evidence.empty and 'earliest_high_timestamp' in tomcat01_trace_evidence.columns:
    tomcat01_trace_evidence = tomcat01_trace_evidence.sort_values('earliest_high_timestamp', ascending=True).reset_index(drop=True)
tomcat01_trace_evidence_display = tomcat01_trace_evidence[['cmdb_id','trace_name','earliest_high_timestamp','max_value_in_window','severity_high']].head(20)

# 3) Error logs: earliest error rows for apache01 and apache02 (earliest 3 rows each)
if not error_logs_window.empty:
    apache_error_full = error_logs_window[error_logs_window['cmdb_id'].isin(['apache01','apache02'])].copy()
    apache_error_full = apache_error_full.sort_values('timestamp', ascending=True).reset_index(drop=True)
    # Store full result as requested
    apache_error_earliest = apache_error_full.copy()
    # For display: earliest 3 rows per cmdb_id
    apache_error_earliest_display = apache_error_full.groupby('cmdb_id', sort=False).head(3)[['timestamp','cmdb_id','message']].reset_index(drop=True)
else:
    apache_error_earliest = pd.DataFrame(columns=['timestamp','cmdb_id','message'])
    apache_error_earliest_display = apache_error_earliest.copy()

# Final displayed outputs (bounded):
tomcat01_metric_evidence_display, tomcat01_trace_evidence_display, apache_error_earliest_display

```
Out[10]:
```
```python
# Build a concise evidential summary for Tomcat01 using in-kernel tables:
# tomcat01_metric_evidence_display, tomcat01_trace_evidence_display, apache_error_earliest_display
import pandas as pd

# Safely get variables from kernel
metric_df = tomcat01_metric_evidence_display.copy() if 'tomcat01_metric_evidence_display' in globals() else pd.DataFrame()
trace_df  = tomcat01_trace_evidence_display.copy()  if 'tomcat01_trace_evidence_display' in globals()  else pd.DataFrame()
apache_df = apache_error_earliest_display.copy()    if 'apache_error_earliest_display' in globals() else pd.DataFrame()

def fmt_ts(v):
    if pd.isna(v):
        return "None"
    try:
        return pd.to_datetime(v).tz_convert('Asia/Shanghai').isoformat()
    except Exception:
        try:
            return pd.to_datetime(v).isoformat()
        except Exception:
            return str(v)

def short(s, n=160):
    if pd.isna(s):
        return ""
    s = str(s)
    return s if len(s) <= n else s[:n-1] + "…"

lines = []
lines.append("Tomcat01 evidential summary (compact):")

# 1) Metric anomalies
if metric_df.empty:
    lines.append("- Metrics: no metric anomalies found for Tomcat01 in metric_anom_full.")
else:
    lines.append(f"- Metrics: {len(metric_df)} anomalous series (showing top {min(10,len(metric_df))}):")
    for i, r in metric_df.head(10).iterrows():
        kpi = r.get('kpi_name', r.get('series_name', 'unknown'))
        sev = r.get('severity_max', r.get('severity_max', 'None'))
        sev_s = f"{float(sev):.3f}" if (pd.notna(sev) and sev != pd.NA) else "None"
        ear = fmt_ts(r.get('earliest_high_timestamp'))
        cnt = int(r.get('count_high')) if pd.notna(r.get('count_high')) else 0
        mv = r.get('max_value_in_window')
        lines.append(f"  • {kpi} — severity={sev_s}, earliest={ear}, count_high={cnt}, max={mv}")

# 2) Incoming traces targeting Tomcat01
if trace_df.empty:
    lines.append("- Incoming traces: no trace-to-Tomcat01 events found (trace_to_targets / trace_anom_full).")
else:
    lines.append(f"- Incoming traces targeting Tomcat01: {len(trace_df)} row(s) (showing up to 20, sorted earliest first):")
    for i, r in trace_df.head(20).iterrows():
        caller = r.get('cmdb_id', 'unknown')
        tname  = r.get('trace_name')
        ear    = fmt_ts(r.get('earliest_high_timestamp'))
        mv     = r.get('max_value_in_window')
        sev    = r.get('severity_high')
        sev_s  = f"{float(sev):.3f}" if (pd.notna(sev) and sev != pd.NA) else "None"
        lines.append(f"  • from {caller}: {tname} — earliest={ear}, max={mv}, severity={sev_s}")

# 3) Apache error context (earliest 3 rows per apache01/apache02)
if apache_df.empty:
    lines.append("- Apache errors: no error logs for apache01/apache02 in the window.")
else:
    # Group earliest 3 per cmdb_id
    lines.append("- Apache error samples (earliest up to 3 per apache01/apache02):")
    grouped = apache_df.groupby('cmdb_id', sort=False)
    for comp, g in grouped:
        g3 = g.head(3)
        lines.append(f"  • {comp}: {len(g)} row(s) total, earliest samples:")
        for _, row in g3.iterrows():
            ts = fmt_ts(row['timestamp'])
            msg = short(row.get('message',''), 140)
            lines.append(f"      - {ts}: {msg}")

lines.append("End of summary.")
summary = "\n".join(lines)

summary
```

The original code execution output of IPython Kernel is also provided below for reference:

(                                            kpi_name  severity_max   earliest_high_timestamp  count_high  max_value_in_window
0     OSLinux-OSLinux_LOCALDISK_LOCALDISK-sda_DSKBps      3.333333 2021-03-24 11:07:00+08:00           1             1.000000
1                             OSLinux-CPU_CPU_CPUWio      2.519324 2021-03-24 11:13:00+08:00           2             3.432494
2     OSLinux-OSLinux_LOCALDISK_LOCALDISK-sdb_DSKBps      1.764706 2021-03-24 11:23:00+08:00           1             3.000000
3                    OSLinux-CPU_CPU-2_SingleCpuUtil      1.600520 2021-03-24 11:25:00+08:00           1            64.071900
4  Tomcat-Threads_7441-"http-nio-8003"_CurrentThr...      1.142857 2021-03-24 11:12:00+08:00           2            12.000000
5                    OSLinux-CPU_CPU-1_SingleCpuUtil      1.139034 2021-03-24 11:18:00+08:00           1           100.000000
6   OSLinux-OSLinux_LOCALDISK_LOCALDISK-sdb_DSKWrite      1.117540 2021-03-24 11:07:00+08:00           1           284.666667
7                    OSLinux-CPU_CPU-3_SingleCpuUtil      1.114706 2021-03-24 11:18:00+08:00           1             5.516700
8    OSLinux-OSLinux_LOCALDISK_LOCALDISK-sdb_DSKWTps      1.101137 2021-03-24 11:05:00+08:00           1             1.533333
9    OSLinux-OSLinux_LOCALDISK_LOCALDISK-sda_DSKWTps      1.092012 2021-03-24 11:11:00+08:00           1             0.900000,   cmdb_id                       trace_name   earliest_high_timestamp  max_value_in_window  severity_high
0    IG01      trace.to_Tomcat01.row_count 2021-03-24 11:00:00+08:00           337.000000       1.056426
1    IG01  trace.to_Tomcat01.duration_mean 2021-03-24 11:13:00+08:00             1.068717       1.210349
2    IG01   trace.to_Tomcat01.duration_p95 2021-03-24 11:13:00+08:00             5.096000       1.226740,                   timestamp   cmdb_id                                            message
0 2021-03-24 11:08:00+08:00  apache02  IPAddress "POST /UOCP/person/ServiceTest9.json...
1 2021-03-24 11:08:01+08:00  apache02  IPAddress "POST /UOCP/base/ServiceTest6.json H...
2 2021-03-24 11:08:04+08:00  apache01  IPAddress "POST /UOCP/person/ServiceTest9.json...
3 2021-03-24 11:08:05+08:00  apache02  IPAddress "POST /UOCP/base/ServiceTest7.json H...
4 2021-03-24 11:08:06+08:00  apache01  IPAddress "POST /UOCP/person/ServiceTest5.json...
5 2021-03-24 11:08:08+08:00  apache01  IPAddress "POST /UOCP/person/ServiceTest9.json...)```
```

In [ ]:
# In[11]:

import pandas as pd
import pytz

tz = pytz.timezone('Asia/Shanghai')

# Reuse existing variables: metric_anom_full, metric_df, error_logs_window
# Fallback if metric_df not present
try:
    metric_df
except NameError:
    metric_df = pd.read_csv('metric.csv')

# Ensure metric_df has correct types and tz-aware datetime column
metric_df['value'] = pd.to_numeric(metric_df['value'], errors='coerce')
metric_df['timestamp'] = pd.to_numeric(metric_df['timestamp'], errors='coerce').astype('Int64')
metric_df['ts_dt'] = pd.to_datetime(metric_df['timestamp'].astype('int64'), unit='s', utc=True).dt.tz_convert(tz)

# Ensure metric_anom_full exists
if 'metric_anom_full' not in globals():
    metric_anom_full = pd.DataFrame()

# Window boundaries
start = pd.Timestamp('2021-03-24 11:00:00', tz=tz)
end   = pd.Timestamp('2021-03-24 11:30:00', tz=tz)

# 1) Try to pick earliest metric anomaly from metric_anom_full for Tomcat01
tomcat01_earliest_metric_point = pd.DataFrame(columns=['kpi_name','earliest_high_timestamp','severity','max_value_in_window'])

if not metric_anom_full.empty:
    # Normalize timestamp column
    if 'earliest_high_timestamp' in metric_anom_full.columns:
        metric_anom_full['earliest_high_timestamp'] = pd.to_datetime(metric_anom_full['earliest_high_timestamp'], utc=True, errors='coerce').dt.tz_convert(tz)
    # Filter rows for Tomcat01 with count_high>0 and a valid earliest_high_timestamp
    mrows = metric_anom_full[(metric_anom_full.get('cmdb_id') == 'Tomcat01') & (metric_anom_full.get('count_high', 0) > 0)].copy()
    mrows = mrows[mrows['earliest_high_timestamp'].notna()]
    if not mrows.empty:
        # choose row with smallest earliest_high_timestamp
        mrows = mrows.sort_values('earliest_high_timestamp', ascending=True).reset_index(drop=True)
        row = mrows.iloc[0]
        kpi = row.get('kpi_name', row.get('series_name', pd.NA))
        # prefer severity_max, else severity_high
        sev = row.get('severity_max', row.get('severity_high', pd.NA))
        mv = row.get('max_value_in_window', pd.NA)
        tomcat01_earliest_metric_point = pd.DataFrame([{
            'kpi_name': kpi,
            'earliest_high_timestamp': row['earliest_high_timestamp'],
            'severity': sev,
            'max_value_in_window': mv
        }])
#  If no result from metric_anom_full, compute from raw metric_df using global P95 per series
if tomcat01_earliest_metric_point.empty:
    # compute global P95 per Tomcat01 (full series)
    tt = metric_df[metric_df['cmdb_id'] == 'Tomcat01'].copy()
    if not tt.empty:
        gp95 = tt.groupby('kpi_name', sort=False)['value'].quantile(0.95).rename('global_P95').reset_index()
        # join thresholds back
        tt = tt.merge(gp95, on='kpi_name', how='left')
        # restrict to failure window
        win = tt[(tt['ts_dt'] >= start) & (tt['ts_dt'] <= end)].copy()
        # find rows where value >= global_P95
        win_high = win[win['value'] >= win['global_P95']].copy()
        if not win_high.empty:
            # find earliest timestamp among these
            win_high = win_high.sort_values('ts_dt', ascending=True).reset_index(drop=True)
            first = win_high.iloc[0]
            kpi = first['kpi_name']
            ear = first['ts_dt']
            # compute severity_high = max_value_in_window / global_P95 for that series within window
            series_window = win[win['kpi_name'] == kpi]
            max_in_window = series_window['value'].max()
            gp95_val = float(first['global_P95']) if pd.notna(first['global_P95']) else pd.NA
            severity = (float(max_in_window) / gp95_val) if (pd.notna(gp95_val) and gp95_val != 0) else pd.NA
            tomcat01_earliest_metric_point = pd.DataFrame([{
                'kpi_name': kpi,
                'earliest_high_timestamp': ear,
                'severity': severity,
                'max_value_in_window': max_in_window
            }])

# Store full intermediate result (even if empty)
# tomcat01_earliest_metric_point variable already set

# 2) Fetch raw metric value(s) at that exact earliest_high_timestamp for Tomcat01 and that kpi_name
tomcat01_earliest_metric_raw = pd.DataFrame(columns=['timestamp','cmdb_id','kpi_name','value'])
if not tomcat01_earliest_metric_point.empty:
    kpi_sel = tomcat01_earliest_metric_point.iloc[0]['kpi_name']
    ts_sel = tomcat01_earliest_metric_point.iloc[0]['earliest_high_timestamp']
    # ensure ts_sel is tz-aware datetime
    if pd.notna(ts_sel):
        # match metric_df rows with same cmdb_id, kpi_name, and ts_dt == ts_sel
        raw_rows = metric_df[
            (metric_df['cmdb_id'] == 'Tomcat01') &
            (metric_df['kpi_name'] == kpi_sel) &
            (metric_df['ts_dt'] == ts_sel)
        ].copy()
        if not raw_rows.empty:
            tomcat01_earliest_metric_raw = raw_rows[['ts_dt','cmdb_id','kpi_name','value']].rename(columns={'ts_dt':'timestamp'}).reset_index(drop=True)
        else:
            # possible floating timestamp mismatch: allow matching by timestamp numeric equality
            raw_rows2 = metric_df[
                (metric_df['cmdb_id'] == 'Tomcat01') &
                (metric_df['kpi_name'] == kpi_sel) &
                (metric_df['timestamp'] == int(ts_sel.tz_convert('UTC').timestamp()))
            ].copy()
            if not raw_rows2.empty:
                tomcat01_earliest_metric_raw = raw_rows2[['ts_dt','cmdb_id','kpi_name','value']].rename(columns={'ts_dt':'timestamp'}).reset_index(drop=True)
# limit to 10 rows
tomcat01_earliest_metric_raw = tomcat01_earliest_metric_raw.head(10)

# 3) Earliest apache error timestamp for apache01 and apache02 from error_logs_window
apache_earliest_errors = pd.DataFrame(columns=['cmdb_id','earliest_error_timestamp'])
if 'error_logs_window' in globals() and not error_logs_window.empty:
    err = error_logs_window[error_logs_window['cmdb_id'].isin(['apache01','apache02'])].copy()
    if not err.empty:
        # ensure timestamp dtype
        err['timestamp'] = pd.to_datetime(err['timestamp'], utc=True, errors='coerce').dt.tz_convert(tz)
        agg = err.groupby('cmdb_id', sort=False).agg(earliest_error_timestamp=('timestamp','min')).reset_index()
        apache_earliest_errors = agg[['cmdb_id','earliest_error_timestamp']]
# limit to 2 rows (apache01, apache02)
apache_earliest_errors = apache_earliest_errors.head(2)

# Display the compact results: metric summary row, up to 10 raw metric rows, and up to 2 apache earliest error rows
tomcat01_earliest_metric_point, tomcat01_earliest_metric_raw, apache_earliest_errors

```
Out[11]:
```
```python
# Summarize findings for Tomcat01 in plain English using in-kernel variables.
# Uses: tomcat01_earliest_metric_point, tomcat01_earliest_metric_raw, apache_earliest_errors

import pandas as pd

# Safety checks and extraction
mp = tomcat01_earliest_metric_point.copy() if 'tomcat01_earliest_metric_point' in globals() else pd.DataFrame()
raw = tomcat01_earliest_metric_raw.copy() if 'tomcat01_earliest_metric_raw' in globals() else pd.DataFrame()
ae = apache_earliest_errors.copy() if 'apache_earliest_errors' in globals() else pd.DataFrame()

if mp.empty:
    summary = "No earliest metric anomaly found for Tomcat01 within the failure window."
else:
    kpi = mp.iloc[0].get('kpi_name', 'Unknown KPI')
    ear = mp.iloc[0].get('earliest_high_timestamp')
    sev = mp.iloc[0].get('severity')
    mv  = mp.iloc[0].get('max_value_in_window')
    ear_s = ear.isoformat() if pd.notna(ear) else "None"
    sev_s = f"{float(sev):.3f}" if (pd.notna(sev) and sev is not pd.NA) else str(sev)
    mv_s = str(mv)

    # Raw metric rows (up to 10)
    raw_lines = []
    if raw.empty:
        raw_lines.append("No raw metric points found at that timestamp.")
    else:
        for _, r in raw.iterrows():
            ts = r['timestamp'].isoformat() if pd.notna(r['timestamp']) else "None"
            val = r['value']
            raw_lines.append(f"{ts} | {r['cmdb_id']} | {r['kpi_name']} = {val}")

    # Apache earliest errors (apache02 / apache01)
    apache_lines = []
    if ae.empty:
        apache_lines.append("No apache errors in-window for apache01/apache02.")
    else:
        for _, r in ae.iterrows():
            ts = r['earliest_error_timestamp'].isoformat() if pd.notna(r['earliest_error_timestamp']) else "None"
            apache_lines.append(f"{r['cmdb_id']}: {ts}")

    # Build summary string
    lines = [
        f"Earliest Tomcat01 metric anomaly:",
        f"  - KPI: {kpi}",
        f"  - Earliest timestamp: {ear_s}",
        f"  - Severity: {sev_s}",
        f"  - Max value in window: {mv_s}",
        "",
        "Raw metric point(s) at that timestamp (up to 10):"
    ]
    lines += [f"  - {ln}" for ln in raw_lines]
    lines += ["", "Earliest apache error timestamps (apache02 / apache01):"]
    lines += [f"  - {ln}" for ln in apache_lines]

    summary = "\n".join(lines)

summary
```

The original code execution output of IPython Kernel is also provided below for reference:

(                                            kpi_name   earliest_high_timestamp  severity  max_value_in_window
0  Tomcat-Sessions_7441--UOCP_SESSIONRejectedSess... 2021-03-24 11:00:00+08:00       0.0                  0.0,                   timestamp   cmdb_id                                           kpi_name  value
0 2021-03-24 11:00:00+08:00  Tomcat01  Tomcat-Sessions_7441--UOCP_SESSIONRejectedSess...    0.0,     cmdb_id  earliest_error_timestamp
0  apache02 2021-03-24 11:08:00+08:00
1  apache01 2021-03-24 11:08:04+08:00)```
```

In [ ]:
# In[12]:

import pandas as pd
import pytz

tz = pytz.timezone('Asia/Shanghai')

# Reuse metric_df if present; otherwise load
try:
    metric_df
except NameError:
    metric_df = pd.read_csv('metric.csv')

# Ensure types and tz-aware datetime column (UTC -> Asia/Shanghai)
metric_df['value'] = pd.to_numeric(metric_df['value'], errors='coerce')
metric_df['timestamp'] = pd.to_numeric(metric_df['timestamp'], errors='coerce').astype('Int64')
metric_df['ts_dt'] = pd.to_datetime(metric_df['timestamp'].astype('int64'), unit='s', utc=True).dt.tz_convert(tz)

# 1) Identify Tomcat01 KPIs containing 'LOCALDISK' or 'DSK' (case-insensitive)
mask_tomcat = metric_df['cmdb_id'] == 'Tomcat01'
mask_name = metric_df['kpi_name'].str.contains('LOCALDISK|DSK', case=False, regex=True, na=False)
tomcat_disk_kpis = metric_df[mask_tomcat & mask_name][['kpi_name']].drop_duplicates().reset_index(drop=True)

# If none found, produce empty result
if tomcat_disk_kpis.empty:
    tomcat01_disk_breaches = pd.DataFrame(columns=['timestamp','cmdb_id','kpi_name','value','global_P95'])
    tomcat01_disk_breaches_display = tomcat01_disk_breaches.head(20)
else:
    # 2) Compute global P95 per discovered KPI using the full series (DO NOT filter by time before computing)
    kpi_list = tomcat_disk_kpis['kpi_name'].tolist()
    full_relevant = metric_df[metric_df['kpi_name'].isin(kpi_list)].copy()
    gp95 = full_relevant.groupby(['cmdb_id','kpi_name'], sort=False)['value'].quantile(0.95).rename('global_P95').reset_index()

    # 3) Filter points to narrow window 11:00:00 through 11:10:00 inclusive (UTC+8)
    start = pd.Timestamp('2021-03-24 11:00:00', tz=tz)
    end   = pd.Timestamp('2021-03-24 11:10:00', tz=tz)
    window_pts = metric_df[
        (metric_df['cmdb_id'] == 'Tomcat01') &
        (metric_df['kpi_name'].isin(kpi_list)) &
        (metric_df['ts_dt'] >= start) &
        (metric_df['ts_dt'] <= end)
    ].copy()

    # Merge global P95 into window points
    window_with_p95 = window_pts.merge(gp95, on=['cmdb_id','kpi_name'], how='left')

    # Keep only rows where value >= global_P95 and global_P95 is not null
    # If global_P95 is NaN, treat as no threshold -> exclude
    breaches = window_with_p95[window_with_p95['global_P95'].notna() & (window_with_p95['value'] >= window_with_p95['global_P95'])].copy()

    # Select required columns and sort by timestamp ascending
    tomcat01_disk_breaches = breaches[['ts_dt','cmdb_id','kpi_name','value','global_P95']].rename(columns={'ts_dt':'timestamp'}).sort_values('timestamp', ascending=True).reset_index(drop=True)

    # Limit displayed output to at most 20 rows
    tomcat01_disk_breaches_display = tomcat01_disk_breaches.head(20)

# Store full result in kernel variable (already assigned) and display limited table
tomcat01_disk_breaches, tomcat01_disk_breaches_display

```
Out[12]:
```
```python
# Summarize Tomcat01 disk-I/O breach points (compact plain English)
# Uses in-kernel variable: tomcat01_disk_breaches (full breaches stored previously)

import pandas as pd

# Safety: load variable if exists
df = tomcat01_disk_breaches.copy() if 'tomcat01_disk_breaches' in globals() else pd.DataFrame()

if df.empty:
    summary = "No Tomcat01 disk breaches found in the specified narrow window (2021-03-24 11:00–11:10 UTC+8)."
else:
    # Ensure timestamp is datetime tz-aware and convert to string
    df['timestamp'] = pd.to_datetime(df['timestamp'], utc=True, errors='coerce').dt.tz_convert('Asia/Shanghai')
    total_breaches = int(df.shape[0])
    earliest_ts = df['timestamp'].min()
    earliest_s = earliest_ts.isoformat() if pd.notna(earliest_ts) else "None"

    # Count breaches where global_P95 == 0 (these may reflect zero-threshold series)
    zero_p95_count = int((df['global_P95'] == 0).sum())
    nonzero_df = df[df['global_P95'] > 0].copy()
    nonzero_count = int(nonzero_df.shape[0])

    # Compute severity for nonzero thresholds and show top 3 by severity
    if not nonzero_df.empty:
        nonzero_df['severity'] = nonzero_df['value'] / nonzero_df['global_P95']
        top3 = nonzero_df.sort_values('severity', ascending=False).head(3)
        top_lines = []
        for _, r in top3.iterrows():
            ts = r['timestamp'].isoformat()
            kpi = r['kpi_name']
            val = r['value']
            p95 = r['global_P95']
            sev = float(r['severity'])
            top_lines.append(f"{ts} — {kpi}: value={val}, global_P95={p95}, severity={sev:.3f}")
        top_summary = " | ".join(top_lines)
    else:
        top_summary = "No breaches with non-zero global_P95."

    summary_lines = [
        f"Tomcat01 disk-I/O breach summary (narrow window 2021-03-24 11:00–11:10 UTC+8):",
        f"- Total breach rows found: {total_breaches}",
        f"- Earliest breach timestamp: {earliest_s}",
        f"- Breaches with global_P95==0 (zero-threshold series): {zero_p95_count}",
        f"- Breaches with global_P95>0: {nonzero_count}",
        f"- Top non-zero-threshold breaches by severity (up to 3): {top_summary}",
        f"- Note: examples include sda_DSKRead at 2021-03-24 11:01 (value=6.0, global_P95=6.0) and sdb_DSKWrite at 2021-03-24 11:07 (value≈284.667, global_P95≈284.233)."
    ]
    summary = "\n".join(summary_lines)

summary
```

The original code execution output of IPython Kernel is also provided below for reference:

(                   timestamp   cmdb_id                                           kpi_name     value  global_P95
0  2021-03-24 11:01:00+08:00  Tomcat01    OSLinux-OSLinux_LOCALDISK_LOCALDISK-sda_DSKRTps  0.183333    0.183333
1  2021-03-24 11:01:00+08:00  Tomcat01    OSLinux-OSLinux_LOCALDISK_LOCALDISK-sda_DSKRead  6.000000    6.000000
2  2021-03-24 11:01:00+08:00  Tomcat01    OSLinux-OSLinux_LOCALDISK_LOCALDISK-sdb_DSKRTps  0.000000    0.000000
3  2021-03-24 11:01:00+08:00  Tomcat01    OSLinux-OSLinux_LOCALDISK_LOCALDISK-sdb_DSKRead  0.000000    0.000000
4  2021-03-24 11:03:00+08:00  Tomcat01  OSLinux-OSLinux_LOCALDISK_LOCALDISK-sda_DSKAvg...  0.860000    0.860000
..                       ...       ...                                                ...       ...         ...
19 2021-03-24 11:08:00+08:00  Tomcat01  OSLinux-OSLinux_LOCALDISK_LOCALDISK-sda_DSKAvg...  0.860000    0.860000
20 2021-03-24 11:08:00+08:00  Tomcat01  OSLinux-OSLinux_LOCALDISK_LOCALDISK-sdb_DSKAvg...  0.860000    0.860000
21 2021-03-24 11:09:00+08:00  Tomcat01     OSLinux-OSLinux_LOCALDISK_LOCALDISK-sda_DSKTps  0.000000    0.000000
22 2021-03-24 11:09:00+08:00  Tomcat01    OSLinux-OSLinux_LOCALDISK_LOCALDISK-sdb_DSKRTps  0.000000    0.000000
23 2021-03-24 11:09:00+08:00  Tomcat01    OSLinux-OSLinux_LOCALDISK_LOCALDISK-sdb_DSKRead  0.000000    0.000000

[24 rows x 5 columns],                    timestamp   cmdb_id                                           kpi_name       value  global_P95
0  2021-03-24 11:01:00+08:00  Tomcat01    OSLinux-OSLinux_LOCALDISK_LOCALDISK-sda_DSKRTps    0.183333    0.183333
1  2021-03-24 11:01:00+08:00  Tomcat01    OSLinux-OSLinux_LOCALDISK_LOCALDISK-sda_DSKRead    6.000000    6.000000
2  2021-03-24 11:01:00+08:00  Tomcat01    OSLinux-OSLinux_LOCALDISK_LOCALDISK-sdb_DSKRTps    0.000000    0.000000
3  2021-03-24 11:01:00+08:00  Tomcat01    OSLinux-OSLinux_LOCALDISK_LOCALDISK-sdb_DSKRead    0.000000    0.000000
4  2021-03-24 11:03:00+08:00  Tomcat01  OSLinux-OSLinux_LOCALDISK_LOCALDISK-sda_DSKAvg...    0.860000    0.860000
5  2021-03-24 11:03:00+08:00  Tomcat01     OSLinux-OSLinux_LOCALDISK_LOCALDISK-sda_DSKTps    0.000000    0.000000
6  2021-03-24 11:03:00+08:00  Tomcat01  OSLinux-OSLinux_LOCALDISK_LOCALDISK-sdb_DSKAvg...    0.860000    0.860000
7  2021-03-24 11:03:00+08:00  Tomcat01  OSLinux-OSLinux_LOCALDISK_LOCALDISK-sdb_DSKPer...    0.300000    0.300000
8  2021-03-24 11:03:00+08:00  Tomcat01    OSLinux-OSLinux_LOCALDISK_LOCALDISK-sdb_DSKRTps    0.000000    0.000000
9  2021-03-24 11:03:00+08:00  Tomcat01    OSLinux-OSLinux_LOCALDISK_LOCALDISK-sdb_DSKRead    0.000000    0.000000
10 2021-03-24 11:05:00+08:00  Tomcat01    OSLinux-OSLinux_LOCALDISK_LOCALDISK-sdb_DSKWTps    1.533333    1.392500
11 2021-03-24 11:05:00+08:00  Tomcat01    OSLinux-OSLinux_LOCALDISK_LOCALDISK-sdb_DSKRead    0.000000    0.000000
12 2021-03-24 11:05:00+08:00  Tomcat01    OSLinux-OSLinux_LOCALDISK_LOCALDISK-sdb_DSKRTps    0.000000    0.000000
13 2021-03-24 11:05:00+08:00  Tomcat01     OSLinux-OSLinux_LOCALDISK_LOCALDISK-sda_DSKTps    0.000000    0.000000
14 2021-03-24 11:07:00+08:00  Tomcat01     OSLinux-OSLinux_LOCALDISK_LOCALDISK-sda_DSKBps    1.000000    0.300000
15 2021-03-24 11:07:00+08:00  Tomcat01  OSLinux-OSLinux_LOCALDISK_LOCALDISK-sdb_DSKPer...    0.300000    0.300000
16 2021-03-24 11:07:00+08:00  Tomcat01    OSLinux-OSLinux_LOCALDISK_LOCALDISK-sdb_DSKRTps    0.000000    0.000000
17 2021-03-24 11:07:00+08:00  Tomcat01    OSLinux-OSLinux_LOCALDISK_LOCALDISK-sdb_DSKRead    0.000000    0.000000
18 2021-03-24 11:07:00+08:00  Tomcat01   OSLinux-OSLinux_LOCALDISK_LOCALDISK-sdb_DSKWrite  284.666667  284.233333
19 2021-03-24 11:08:00+08:00  Tomcat01  OSLinux-OSLinux_LOCALDISK_LOCALDISK-sda_DSKAvg...    0.860000    0.860000)

**Note**: The printed pandas DataFrame is truncated due to its size. Only **10 rows** are displayed, which may introduce observation bias due to the incomplete table. If you want to comprehensively understand the details without bias, please ask Executor using `df.head(X)` to display more rows.```
```